# Elliptic++ — SAFE-G (aşama E6)

Gerçek, graf-yerli ve zamansal ikinci ana veri seti. Görev: **işlem (tx) düzeyinde illicit tespiti**.
Varlık: işlemin **giriş cüzdanları** (gönderen); karşı taraf: **çıkış cüzdanları**.

Bu aşamada: veri yükleme ve şema kontrolü, κ önizlemesi, hazır cüzdan özniteliklerinin zaman denetimi,
fold-wise öznitelik hesaplama, artefakt denetimi, sızıntı öz-testi ve standart zamansal bölmede (1–34 / 35–49) hızlı kontrol.
Sonraki aşama (E2): P1 / P3 / rolling-origin tam grid, κ-duyarlı eğitim ve router, istatistik, GNN karşılaştırması.

**Veri:** Elmougy & Liu (KDD 2023) — github.com/git-disl/EllipticPlusPlus (Google Drive). Gerekli dosyalar:
`txs_features.csv`, `txs_classes.csv`, `txs_edgelist.csv`, `AddrTx_edgelist.csv`, `TxAddr_edgelist.csv`;
isteğe bağlı: `wallets_features.csv` (yalnızca zaman denetimi için). İşlem kısmı orijinal Elliptic (Weber vd. 2019) ile aynıdır.


## Revizyon günlüğü

| Kod | Karar / değişiklik | Gerekçe |
|---|---|---|
| **E1** | Elliptic++ ikinci ana veri seti olarak eklendi (danışman kararı). Görev işlem düzeyinde; varlık giriş cüzdanları, karşı taraf çıkış cüzdanları. | Gerçek veri, graf-yerli, 49 zaman adımı; cüzdanlar adımlar arasında tekrar ediyor → κ anlamlı. |
| **E1** | Etiketsiz işlemler (class 3) eğitim/değerlendirme satırı değil, yalnızca **yapı** (graf ve cüzdan geçmişi) için kullanılıyor; yapı bölümü protokole göre: zamansal protokollerde adım ≤ kesim, rastgele CV'de değerlendirme işlemleri hariç tüm işlemler. | Etiket sızıntısı yok; yapı bilgisi tahmin anında gözlenebilir olanla sınırlı. |
| **E1** | tx–tx grafı öznitelikleri (adım içi PageRank, bileşen boyutu, Elliptic++'ın derece öznitelikleri) fold'dan bağımsız hesaplanıyor. | Elliptic'te tx–tx kenarları yalnızca aynı adım içinde (notebook'ta doğrulanıyor); bir işlemin kendi adımındaki yapısı etiket içermez ve tahmin anında gözlenir. Graf öznitelikleri setinde, Set A dışında. |
| **E1** | Cüzdan öznitelikleri (gönderim/alım sayısı, aktif adım sayısı, yaş, PageRank, OOF fraud oranı) fold'un yapı bölümünden **bizim tarafımızdan** hesaplanıyor; Elliptic++'ın hazır `wallets_features.csv` öznitelikleri kullanılmıyor. | Hazır cüzdan öznitelikleri (ör. `lifetime_in_blocks`, `num_timesteps_appeared_in`) tüm zaman çizelgesi üzerinden hesaplanmış görünüyor → erken adımlarda gelecek bilgisi. Notebook'ta denetleniyor. |
| **E1** | Topluluk (Louvain) yerine cüzdan düzeyinde OOF fraud oranı (guilt-by-association). | ~1M düğümlü cüzdan–işlem grafında Louvain fold başına çok pahalı; cüzdan düzeyi oran aynı bilgiyi daha doğrudan taşıyor. |
| **E1 sonuç** | κ (giriş cüzdanı): P3 = 0.302; rolling 24/29/34/39/44 → 0.415 / 0.352 / 0.396 / **0.248** / 0.480. Cüzdanların %92.5'i tek adımda görünüyor (ort. 1.12 adım). | Bitcoin adresleri büyük ölçüde tek kullanımlık; Elliptic'te κ doğal olarak düşük. Origin 39 (test 40–44) dark market kapanmasını kapsıyor. |
| **E1 sonuç** | Hazır cüzdan öznitelikleri: çok adımlı adreslerde ömür boyu değerler tüm satırlarda %100 aynı; adresin ilk göründüğü adımın satırında bile %22.5'inde `num_timesteps_appeared_in > 1`. | Elliptic++'ın hazır cüzdan öznitelikleri tüm zaman çizelgesinden hesaplanmış → **zamansal sızıntı doğrulandı**; kullanılmıyor. |
| **E1 sonuç** | Denetim: C1 geçti (a* = 0.866, `size`); **C2 kaldı** (F1* = 0.774, iki yerel öznitelik kuralı, precision 0.96); **C3 kaldı** (ρ1 = 0.933); **C4 kaldı** (λ = 1.26); C5 geçti. | Elliptic'te yerel öznitelikler kısa yola yakın ayırıcı; cüzdan grafı varlık geçmişi açısından dejenere (tek kullanımlık adresler). |
| **E1 sonuç** | P3 hızlı kontrol: LightGBM Set A F1 0.758 / PR-AUC 0.789, A+AGG 0.797 / 0.798 (literatürle uyumlu). Görülen cüzdanlı test işlemlerinde graf öznitelikleri PR-AUC'yi ~0.01'den 0.21–0.41'e çıkarıyor; görülmeyenlerde 0.01–0.05 düşürüyor. Toplam metrik bu iki etkiyi gizliyor. LR'de B ve C setlerinde F1 ≈ 0 (eşik transferi çöküşü). | κ'ya göre ayrılmış raporlamanın ve router'ın doğrudan gerekçesi. |
| **E1** | Protokol 2 (varlık gruplu) tanımlanmadı. | Bir işlemin birden çok giriş cüzdanı var; varlık gruplaması tek anlamlı değil. κ çeşitliliği zamansal ve rolling-origin protokollerinden geliyor. |
| **E2** | Tam grid: P1 (rastgele 5-fold, önceki çalışmalardaki kurgu), P3 (standart 1–34 / 35–49), P4 rolling-origin (`ROLLING_ORIGINS`, ufuk `ROLLING_HORIZON`). Her fold'da A, B, C, D, A+AGG + κ-mask (C, D) + router (C|A, D|A). Tahminler kaydediliyor. | Editörün "tek zamansal bölme" eleştirisi; drift (origin 39) ve κ çeşitliliği. |
| **E2** | κ-duyarlı eğitim burada **iki taraflı**: giriş ve çıkış cüzdanı öznitelikleri, kendi κ̂'larına göre ayrı ayrı maskeleniyor (`is_cold_ent`, `is_cold_cp`). Maskeleme birimi işlem (cüzdanlar büyük ölçüde tek kullanımlık). | Elliptic'te karşı taraf da büyük ölçüde görülmemiş (κ_cp ≈ 0.27); IEEE/BankSim'deki "karşı taraf her zaman görülmüş" varsayımı burada geçerli değil. |
| **E2** | κ'ya göre ayrılmış tabloya alt grup prevalansı ve pozitif sayısı eklendi. | Görülen alt gruptaki PR-AUC'nin taban çizgisi (prevalans) ile birlikte okunması gerekiyor. |
| **Karar** | Artefakt denetimi bir "kapı" değil, **önceden tanımlanmış teşhis profili**: eşikler değişmedi; her kriterin kalması yorumu kısıtlar (C1/C2 → ablation kısa yol yakalama hızını ölçer; C3/C4 → graf için boş sonuç bilgilendirici değil). Hiçbir veri seti atılmıyor; tüm sonuçlar profiliyle birlikte raporlanıyor. | Üç benchmark'ın hiçbiri denetimi tam geçmiyor (BankSim C1/C2, IEEE-CIS uid C3, Elliptic++ C2/C3/C4); bu bir bulgu olarak raporlanıyor. |
| **E2 sonuç** | **Protokol işareti çeviriyor:** P1 (rastgele CV, κ = 0.81): F1 ≈ 0.96, D–A +0.053 F1 / +0.065 PR-AUC, 4/4 modelde anlamlı (düzeltilmiş t + Holm). P3 (κ = 0.30): D–A **−0.082 F1**, PR-AUC +0.012. P4 rolling (κ ort. 0.38): D–A −0.048 F1, +0.015 PR-AUC; origin 24–34'te PR-AUC +0.04–0.05, origin 39 (dark market) ve 44'te negatif. | Önceki çalışmalardaki rastgele bölme hem mutlak performansı hem de graf katkısını şişiriyor. |
| **E2 sonuç** | **Mekanizma (κ-ayrılmış):** P4'te görülen cüzdanlı test işlemlerinde (prevalans %6, ~72 pozitif/fold) PR-AUC A 0.40 → C 0.73 → C κ-mask 0.78; görülmeyenlerde (prevalans %15) A 0.71 → C 0.67. P3'te görülen alt grupta yalnızca **5 pozitif** var (prevalans %0.1) — bu alt grup sonucu yorumlanmamalı. | Kazanç görülen varlıklarda, kayıp görülmeyenlerde; toplam metrik ikisini birbirine karıştırıyor. |
| **E2 sonuç** | **Çözüm katmanı:** Router C\|A ortalamada en iyi: P3 PR-AUC 0.682 (A 0.659, C 0.662), F1 0.688 (A 0.674); P4 PR-AUC 0.732 (A 0.689, C 0.692), F1 0.702 (A 0.671). κ-mask C'yi her protokolde iyileştiriyor (P3 F1 +0.136, P4 +0.093 — büyük kısmı LR'nin eşik çöküşünün giderilmesi), görülmeyen alt grupta kaybı azaltıyor ama A'ya ulaşmıyor. Elliptic'in hazır komşuluk aggregate'leri (A+AGG) ağaç modellerinde zamansal protokollerde en güçlü set (P3 PR-AUC 0.80). | Router ve κ-mask'ın ikisi de raporlanmalı; A+AGG'nin üstünlüğü dürüstçe yazılmalı. |
| **E3** | GNN karşılaştırması: GCN ve GraphSAGE (2 katman, gizli boyut 64), tx–tx grafında, A ve A+AGG öznitelikleriyle; P3, P4, P1; 3 tohum. Eğitim grafı yalnızca yapı bölümündeki işlemler; eşik ve erken durdurma eğitim içi %20 doğrulamada. κ-ayrılmış metrikler ve gecikme ölçümü. | Editörün "GNN karşılaştırması yok" eleştirisi; aynı protokol ve raporlama ile. |
| **E3 sonuç** | **GNN'ler tablo modellerin gerisinde, her protokolde.** PR-AUC — P1: GraphSAGE (A+AGG) 0.944 vs tablo en iyi 0.992; P3: 0.719 vs 0.798; P4: 0.719 vs 0.851. GraphSAGE > GCN her yerde; tohumlar arası std < 0.03. GNN de P1→P3 düşüşünü gösteriyor (0.944 → 0.719). | Weber vd. (2019) ile uyumlu (RF > GCN). Editörün GNN eleştirisine cevap: aynı sızıntısız protokolde koşuldu. |
| **E3 sonuç** | κ-ayrılmış: P4'te görülen cüzdanlı işlemlerde GNN PR-AUC ≈ 0.42 = tablo Set A (0.40); cüzdan öznitelikleriyle 0.73–0.78. Homojen tx–tx GNN cüzdan geçmişini görmüyor. P3 görülen alt grubu (~5 pozitif) yorumlanmaz. | Heterojen (cüzdan–işlem) GNN kısıt olarak yazılacak. |
| **E3 sonuç** | Gecikme: GNN çıkarımı 0.07–0.6 ms / 1k işlem (GPU, tam graf, graf kurulumu hariç); LightGBM C 86 ms / 1k (süre cüzdan özniteliklerinin tüm fold için toplu hesabı). Ölçümler eşdeğer değil → makalede "ikisi de gerçek zamanlı sınırda" denecek. | Hız karşılaştırmasının dürüst çerçevesi. |
| **E4** | Graf öznitelikleri **güçlü baseline (A+AGG)** üzerine: A+AGG+B, A+AGG+C, A+AGG+D; κ-mask (A+AGG+C, A+AGG+D); router (A+AGG+C \| A+AGG, A+AGG+D \| A+AGG); P1, P3, P4; aynı modeller (`E2_MODELS`). Etiketler `E4_P*`; E2 checkpoint'leri ayrı. κ-ayrılmış tablo alt küme boyutu, pozitif sayısı ve güvenilirlik bayrağıyla (`KAPPA_MIN_POS`). P1'de düzeltilmiş t-testi + Holm; P4 origin bazında (kazanan origin sayısı). | E2'de zamansal protokollerde en güçlü set A+AGG idi (P4 XGBoost PR-AUC 0.851 vs Router C\|A 0.787): önerilen katmanın güçlü baseline'a katkısı ölçülmemişti. |
| **E4** | E3 GNN koşusu varsayılan kapalı (`RUN_E3 = False`): checkpoint varsa sonuçlar yüklenir, yoksa karşılaştırma tablosu atlanır. Gecikme ölçümü kapalı. | E4 için GNN'in yeniden koşulmasına gerek yok. |
| **E5** | P4 üzerinde **kalibre router** (her model, eşiğin seçildiği iç doğrulama bölümünde Platt ölçeklemesiyle kalibre edilir; isotonic karşılaştırma için) ve **geçmiş eşikli router** (giriş cüzdanlarının yapı bölümündeki gönderim + alım sayısı ≥ k ise graf-koşullu model; k ∈ {1, 2, 3, 5, 10}). Dar (C \| A) ve güçlü (A+AGG+C \| A+AGG) baseline. Etiket `E5_P4`; ham-skor router'ın E2/E4 sonuçlarıyla birebir aynı çıktığı otomatik kontrol edilir. | DAMI editörü ve iç değerlendirme: router "heuristic" ve ikili; kalibre edilmemiş skorların birleştirilmesi. |
| **E6** | İkinci hakem turu (P4): (1) **kapasite** — XGBoost ve LightGBM için erken durdurmalı büyük topluluk (en çok `E6_ES_MAX` ağaç); (2) **karşı taraf router'ı** — giriş cüzdanı görülmeyen işlem → taban + çıkış cüzdanı ve işlem grafı öznitelikleri; (3) **prefix (as-of) eğitim öznitelikleri** — eğitim işlemlerinin cüzdan öznitelikleri yalnızca kendi adımından önceki işlemler ve etiketlerle, `E6_PREFIX_REFRESH` adımda bir yenilenen yapıyla (PageRank dahil); (4) origin bazında eşleştirilmiş bootstrap. Etiket `E6_P4`; referans varyant E5 ile birebir kontrol edilir. | Hakem: Elliptic++'ta kapasite, karşı taraf bilgisinin atılması, eğitim özniteliklerinin zamansal geçerliliği, router belirsizliği. |


## 1. Kurulum ve yapılandırma

Colab'da veriyi indirmek için (klasör ~1–2 GB; yalnızca gerekli dosyaları da indirebilirsiniz):
```
# !pip -q install gdown
# !gdown --folder https://drive.google.com/drive/folders/1MRPXz79Lu_JGLlJ21MDfML44dKN9R08l -O elliptic_pp
```


In [1]:
!pip -q install gdown

In [2]:
!gdown --folder https://drive.google.com/drive/folders/1MRPXz79Lu_JGLlJ21MDfML44dKN9R08l -O elliptic_pp

Retrieving folder contents
Processing file 1x5JdNWX8fVM3FO8I0wOLDAeh2qaB53uV AddrAddr_edgelist.csv
Processing file 16eWZTe-dsjEqgsOqFtg78m4fYhcxKlmZ AddrTx_edgelist.csv
Processing file 1SYpun0DMDt-h3sbI60G55jNsfwnFTHXg TxAddr_edgelist.csv
Processing file 1DiBxn8TXdbJqoSw58pYUeaqO3oOKhuQO txs_classes.csv
Processing file 1Q2yG_CIDvfdGP-fKVPSw979EYgQukjz5 txs_edgelist.csv
Processing file 19q09IFhfkOOBOXvn_dKhWjILJtjCcsjc txs_features.csv
Processing file 1ZaACVE4wSIx7r8Z9ze7ExQnJ0wzGrVkp wallets_classes.csv
Processing file 1YLCIn8rPAh4FjKiZynblU2rLx-Fxr2WA wallets_features_classes_combined.csv
Processing file 1mhrrobYdnaxYBIVK06EpKB2EwV-ERlce wallets_features.csv
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1x5JdNWX8fVM3FO8I0wOLDAeh2qaB53uV
From (redirected): https://drive.google.com/uc?id=1x5JdNWX8fVM3FO8I0wOLDAeh2qaB53uV&confirm=t&uuid=ca154017-d4ba-4f1f-a49e-a6c47a0

In [3]:
# pip install pandas numpy networkx scipy scikit-learn xgboost lightgbm matplotlib

import pandas as pd, numpy as np, networkx as nx
import time, warnings, gc, json, os
import matplotlib.pyplot as plt
from scipy import stats, sparse
from scipy.sparse.csgraph import connected_components

from sklearn.model_selection import StratifiedKFold, KFold, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, matthews_corrcoef,
                             confusion_matrix, precision_recall_curve, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier)
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')

# ==================== YAPILANDIRMA ====================
SEED      = 42
N_FOLDS   = 5
FAST_MODE = True
KNN_TRAIN_CAP = 50000

ELLIPTIC_DIR = 'elliptic_pp'          # indirilen klasor; alt klasorler (Transactions Dataset/...) olabilir
FILE_NAMES = {
    'tx_features'    : 'txs_features.csv',
    'tx_classes'     : 'txs_classes.csv',
    'tx_edges'       : 'txs_edgelist.csv',
    'addr_tx'        : 'AddrTx_edgelist.csv',     # input_address -> txId
    'tx_addr'        : 'TxAddr_edgelist.csv',     # txId -> output_address
    'wallet_features': 'wallets_features.csv',    # yalnizca zaman denetimi (istege bagli)
}
ILLICIT_CLASS, LICIT_CLASS = 1, 2     # 3 = bilinmeyen (etiketsiz)

TEMPORAL_CUT    = 34                  # Weber vd. (2019) standart bolme: egitim 1-34, test 35-49
ROLLING_ORIGINS = [24, 29, 34, 39, 44]
ROLLING_HORIZON = 5

PAGERANK_ALPHA = 0.85
PAGERANK_ITERS = 50
TE_SMOOTHING   = 20.0                 # cuzdan fraud orani yumusatmasi
TE_N_SPLITS    = 5                    # egitim satirlari icin ic OOF bolme sayisi

PIPELINE_VERSION   = 'E6'
IMPUTE_POLICY      = 'row_median'
CHECKPOINT_VERSION = 'E1'
CHECKPOINT_DIR     = f'checkpoints_elliptic_{CHECKPOINT_VERSION}'
RESUME = True

USE_DRIVE = True   # Colab oturumlari arasinda checkpoint'leri korumak icin True (E2 grid'i yeniden kosmamak icin)
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    CHECKPOINT_DIR = f'/content/drive/MyDrive/elliptic_checkpoints_{CHECKPOINT_VERSION}'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
np.random.seed(SEED)
pd.set_option('display.width', 200)
print(f'ELLIPTIC_DIR={ELLIPTIC_DIR}  TEMPORAL_CUT={TEMPORAL_CUT}  ROLLING_ORIGINS={ROLLING_ORIGINS}')


Mounted at /content/drive
ELLIPTIC_DIR=elliptic_pp  TEMPORAL_CUT=34  ROLLING_ORIGINS=[24, 29, 34, 39, 44]


## 2. Veri yükleme ve şema kontrolü

In [4]:
import glob, subprocess, sys
_has = lambda f: bool(glob.glob(os.path.join(ELLIPTIC_DIR, '**', f), recursive=True))
if not _has(FILE_NAMES['tx_features']):
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'gdown'], check=True)
    subprocess.run(['gdown', '--folder',
                    'https://drive.google.com/drive/folders/1MRPXz79Lu_JGLlJ21MDfML44dKN9R08l',
                    '-O', ELLIPTIC_DIR], check=True)
print('eksik:', [f for k, f in FILE_NAMES.items() if k != 'wallet_features' and not _has(f)])

eksik: []


In [5]:
def find_file(name, root=ELLIPTIC_DIR):
    hits = sorted(os.path.join(r, f) for r, _, fs in os.walk(root) for f in fs if f == name)
    assert hits, f"'{name}' bulunamadi: '{root}' altinda arandi (ELLIPTIC_DIR'i kontrol edin)"
    return hits[0]

t0 = time.time()
LOCAL_FEATS    = [f'Local_feature_{i}' for i in range(1, 94)]
AGG_FEATS      = [f'Aggregate_feature_{i}' for i in range(1, 73)]
TX_STATS       = (['total_BTC', 'fees', 'size', 'num_input_addresses', 'num_output_addresses']
                  + [f'{d}_BTC_{s}' for d in ('in', 'out') for s in ('min', 'max', 'mean', 'median', 'total')])
TX_GRAPH_GIVEN = ['in_txs_degree', 'out_txs_degree']        # Elliptic++'in verdigi tx-tx dereceleri (graf ozniteligi)
_NUM = LOCAL_FEATS + AGG_FEATS + TX_STATS + TX_GRAPH_GIVEN

tx = pd.read_csv(find_file(FILE_NAMES['tx_features']), dtype={c: np.float32 for c in _NUM})
_missing = [c for c in ['txId', 'Time step'] + _NUM if c not in tx.columns]
assert not _missing, f'Beklenen kolonlar yok ({len(_missing)}): {_missing[:8]}'
_cls = pd.read_csv(find_file(FILE_NAMES['tx_classes']))
assert {'txId', 'class'} <= set(_cls.columns), _cls.columns.tolist()
tx = tx.drop(columns=[c for c in ['class'] if c in tx.columns]).merge(_cls[['txId', 'class']], on='txId', how='left', validate='1:1')
tx = tx.sort_values(['Time step', 'txId']).reset_index(drop=True)
tx['step'] = tx['Time step'].astype(int)
tx['y'] = np.where(tx['class'] == ILLICIT_CLASS, 1.0, np.where(tx['class'] == LICIT_CLASS, 0.0, np.nan))

TX_CODE = pd.Series(np.arange(len(tx)), index=tx['txId'].values)     # txId -> satir kodu
STEP, Y, N_TX = tx['step'].values, tx['y'].values, len(tx)
TOTAL_BTC = tx['total_BTC'].values.astype(np.float64)

def _codes(ids):
    c = TX_CODE.reindex(ids).values
    return c, ~np.isnan(c)

_e = pd.read_csv(find_file(FILE_NAMES['tx_edges'])); assert {'txId1', 'txId2'} <= set(_e.columns)
_a, _ma = _codes(_e['txId1']); _b, _mb = _codes(_e['txId2'])
TX_EDGES = pd.DataFrame({'a': _a[_ma & _mb].astype(int), 'b': _b[_ma & _mb].astype(int)})

_ain  = pd.read_csv(find_file(FILE_NAMES['addr_tx'])); assert {'input_address', 'txId'} <= set(_ain.columns)
_aout = pd.read_csv(find_file(FILE_NAMES['tx_addr'])); assert {'txId', 'output_address'} <= set(_aout.columns)
_wc, WALLETS = pd.factorize(pd.concat([_ain['input_address'], _aout['output_address']], ignore_index=True))
N_W = len(WALLETS)
_ti, _mi = _codes(_ain['txId']); _to, _mo = _codes(_aout['txId'])
E_IN  = pd.DataFrame({'tx': _ti[_mi].astype(int), 'w': _wc[:len(_ain)][_mi]})       # giris cuzdani -> islem
E_OUT = pd.DataFrame({'tx': _to[_mo].astype(int), 'w': _wc[len(_ain):][_mo]})       # islem -> cikis cuzdani
_drop = {'tx_edges': int((~(_ma & _mb)).sum()), 'addr_tx': int((~_mi).sum()), 'tx_addr': int((~_mo).sum())}
del _e, _ain, _aout, _wc, _cls; gc.collect()

LAB = np.where(~np.isnan(Y))[0]
print(f'Yuklendi ({time.time()-t0:.0f}s): {N_TX:,} islem | {len(TX_EDGES):,} tx-tx kenar | {N_W:,} cuzdan | '
      f'{len(E_IN):,} giris + {len(E_OUT):,} cikis kenari')
print(f'Etiketli: {len(LAB):,} (illicit {int(np.nansum(Y)):,}, oran %{np.nanmean(Y)*100:.2f}) | etiketsiz: {N_TX-len(LAB):,}')
print(f'Islem tablosunda olmayan txId nedeniyle atilan kenar: {_drop}')


Yuklendi (19s): 203,769 islem | 234,355 tx-tx kenar | 822,942 cuzdan | 477,117 giris + 837,124 cikis kenari
Etiketli: 46,564 (illicit 4,545, oran %9.76) | etiketsiz: 157,205
Islem tablosunda olmayan txId nedeniyle atilan kenar: {'tx_edges': 0, 'addr_tx': 0, 'tx_addr': 0}


## 3. Yapı kontrolleri ve κ önizlemesi

In [6]:
_cross = float((STEP[TX_EDGES.a.values] != STEP[TX_EDGES.b.values]).mean()) if len(TX_EDGES) else 0.0
print(f'Adimlar ARASI tx-tx kenar orani: %{_cross*100:.3f}  (Elliptic: kenarlar adim icinde -> ~0 beklenir)')

_WS = pd.concat([E_IN, E_OUT], ignore_index=True)
_WS['step'] = STEP[_WS.tx.values]
_nst = _WS.groupby('w')['step'].nunique()
print(f'Birden cok adimda gorunen cuzdan orani: %{(_nst > 1).mean()*100:.1f}  (ortalama {_nst.mean():.2f} adim)')

_has_in = np.zeros(N_TX, bool); _has_in[E_IN.tx.values] = True
print(f'Giris cuzdani olmayan etiketli islem orani: %{(~_has_in[LAB]).mean()*100:.2f}  (bunlar her zaman cold sayilir)')

def kappa_preview(cut, horizon=None):
    """Degerlendirme islemlerinin, en az bir giris cuzdani adim <= cut'ta gorulmus olan orani."""
    seen_w = np.zeros(N_W, bool); seen_w[_WS.w.values[_WS.step.values <= cut]] = True
    ev = (~np.isnan(Y)) & (STEP > cut)
    if horizon: ev &= STEP <= cut + horizon
    e = E_IN[ev[E_IN.tx.values]]
    seen_tx = np.zeros(N_TX, bool); seen_tx[e.tx.values[seen_w[e.w.values]]] = True
    return float(seen_tx[ev].mean()) if ev.any() else np.nan, int(ev.sum())

_k, _n = kappa_preview(TEMPORAL_CUT)
print(f'\nP3 (1-{TEMPORAL_CUT} / {TEMPORAL_CUT+1}-49): kappa = {_k:.3f}  (n_test={_n:,})')
for _o in ROLLING_ORIGINS:
    _k, _n = kappa_preview(_o, ROLLING_HORIZON)
    print(f'  rolling origin {_o:2d} -> test {_o+1}-{min(_o+ROLLING_HORIZON, int(STEP.max()))}: kappa = {_k:.3f} (n={_n:,})')

_cnt = pd.crosstab(tx['step'], tx['class'])
print('\nAdim basina etiket dagilimi (ilk / son 3 adim):'); print(pd.concat([_cnt.head(3), _cnt.tail(3)]).to_string())


Adimlar ARASI tx-tx kenar orani: %0.000  (Elliptic: kenarlar adim icinde -> ~0 beklenir)
Birden cok adimda gorunen cuzdan orani: %7.5  (ortalama 1.12 adim)
Giris cuzdani olmayan etiketli islem orani: %1.11  (bunlar her zaman cold sayilir)

P3 (1-34 / 35-49): kappa = 0.302  (n_test=16,670)
  rolling origin 24 -> test 25-29: kappa = 0.415 (n=2,775)
  rolling origin 29 -> test 30-34: kappa = 0.352 (n=3,513)
  rolling origin 34 -> test 35-39: kappa = 0.396 (n=5,486)
  rolling origin 39 -> test 40-44: kappa = 0.248 (n=7,458)
  rolling origin 44 -> test 45-49: kappa = 0.480 (n=3,726)

Adim basina etiket dagilimi (ilk / son 3 adim):
class   1     2     3
step                 
1      17  2130  5733
2      18  1099  3427
3      11  1268  5342
47     22   824  4275
48     36   435  2483
49     56   420  1978


### Hazır cüzdan özniteliklerinin zaman denetimi
Elliptic++'ın `wallets_features.csv` dosyası her (adres, adım) için bir satır içerir. Ömür boyu öznitelikler
tüm zaman çizelgesinden hesaplanmışsa, bir adresin **ilk göründüğü adımdaki** satırı bile gelecekteki adımları "bilir".

In [7]:
RUN_WALLET_FEATURE_CHECK = False   # E1 sonucu gunlukte; tekrar icin True
WF_CHECK_COLS = ['last_block_appeared_in', 'num_timesteps_appeared_in', 'total_txs', 'lifetime_in_blocks']

if RUN_WALLET_FEATURE_CHECK:
    try:
        _wf = pd.read_csv(find_file(FILE_NAMES['wallet_features']), usecols=['address', 'Time step'] + WF_CHECK_COLS)
    except AssertionError as _e:
        _wf = None; print('wallets_features.csv yok; denetim atlandi.', _e)
    if _wf is not None:
        _multi = _wf[_wf.duplicated('address', keep=False)]
        _const = (_multi.groupby('address')[WF_CHECK_COLS].nunique() == 1).mean()
        _first = _multi.sort_values('Time step').groupby('address').head(1)
        _fut = float((_first['num_timesteps_appeared_in'] > 1).mean())
        wallet_feature_check = pd.DataFrame({
            'olcut': [f'birden cok adimli adreslerde "{c}" tum satirlarda ayni' for c in WF_CHECK_COLS]
                     + ['adresin ILK goruldugu adimdaki satirda num_timesteps_appeared_in > 1'],
            'oran': [round(float(v), 4) for v in _const.values] + [round(_fut, 4)]})
        print(f'Birden cok adimda gorunen adres: {_multi.address.nunique():,}')
        print(wallet_feature_check.to_string(index=False))
        print('\nYorum: oranlar ~1 ise bu oznitelikler tum zaman cizelgesinden hesaplanmistir; erken adim satirlari'
              ' gelecek bilgisi tasir ve SAFE-G kapsaminda KULLANILMAZ.')
        del _wf, _multi; gc.collect()


## 4. Yardımcılar (ortak SAFE-G kodu + Elliptic'e özgü)

In [8]:
# ============================================================
# A1 — OUT-OF-FOLD TARGET ENCODING (community_fraud_rate)
# ------------------------------------------------------------
# Egitim satirlari oranini ic K-fold ile alir: hicbir satir kendi
# etiketini gormez. Test satirlari tum egitim etiketlerinden alir.
# Leave-one-out KULLANILMAZ: LOO orani topluluk icinde satirin kendi
# etiketiyle monoton iliskilidir ve agac modelleri bunu cozebilir.
# ============================================================
from sklearn.model_selection import KFold

TE_N_SPLITS  = 5      # egitim fold'u icindeki ic bolme sayisi
TE_SMOOTHING = 20.0   # m: az islemli topluluklari global oncele ceker (0 = kapali)

def _smoothed_rates(keys, y, m=TE_SMOOTHING):
    """Anahtar bazinda yumusatilmis oran: (sum + m*prior) / (count + m)."""
    prior = float(np.mean(y))
    st = pd.DataFrame({'k': keys, 'y': y}).groupby('k')['y'].agg(['sum', 'count'])
    return ((st['sum'] + m * prior) / (st['count'] + m)).to_dict(), prior

def oof_target_encode(keys_tr, y_tr, keys_te, groups_tr=None,
                      n_splits=TE_N_SPLITS, m=TE_SMOOTHING, seed=SEED):
    """Egitim satirlari: ic K-fold ile OOF (satir kendi etiketini gormez).
    Test satirlari  : tum egitim etiketleri. NaN anahtar (grafta olmayan) -> prior.
    groups_tr verilirse ic bolme GroupKFold olur (ayni varlik ayni ic fold'da)."""
    keys_tr, y_tr = np.asarray(keys_tr, dtype=float), np.asarray(y_tr, dtype=float)
    full_rates, full_prior = _smoothed_rates(keys_tr, y_tr, m)
    te_vals = pd.Series(np.asarray(keys_te, dtype=float)).map(full_rates).fillna(full_prior).values

    if groups_tr is not None:
        k = min(n_splits, pd.Series(groups_tr).nunique())
        it = GroupKFold(n_splits=k).split(keys_tr, y_tr, groups=groups_tr)
    else:
        it = KFold(n_splits=n_splits, shuffle=True, random_state=seed).split(keys_tr)

    tr_vals = np.empty(len(y_tr), dtype=float)
    for a, b in it:
        r, p = _smoothed_rates(keys_tr[a], y_tr[a], m)
        tr_vals[b] = pd.Series(keys_tr[b]).map(r).fillna(p).values
    return tr_vals, te_vals

print(f'A1 OOF target encoding hazır (TE_N_SPLITS={TE_N_SPLITS}, TE_SMOOTHING={TE_SMOOTHING}).')


A1 OOF target encoding hazır (TE_N_SPLITS=5, TE_SMOOTHING=20.0).


In [9]:
# ============================================================
# A2 — GORULMEYEN VARLIKLAR ICIN IMPUTASYON POLITIKASI
# ------------------------------------------------------------
# Egitim grafinda olmayan varliklarin graf ve davranissal oznitelikleri
# NaN olarak uretilir, sonra TEK bir politika ile doldurulur:
#   'row_median' : egitim satirlarindaki (islem agirlikli) medyan. Model egitimde
#                  en sik gordugu degerlerden birini gorur; uc deger uretmez.
#   'legacy'     : A1 ve oncesi davranis (graf -> 0.0, davranissal -> dugum medyani).
#                  Yalnizca karsilastirma / yeniden uretim icin.
# Not: Imputasyon tek basina cold-start sorununu COZMEZ; egitimde hic
# imputed satir yoksa model bu degerleri ogrenmemistir. Cozum katmani
# (kappa-aware egitim, router) ayrica eklenecek.
# ============================================================
IMPUTE_POLICIES = ('row_median', 'legacy')

def impute_values(feat_tr, graph_cols, agg_cols, node_fill, policy=None):
    """Doldurma degerlerini YALNIZCA egitim cercevesinden hesaplar."""
    policy = policy or IMPUTE_POLICY
    if policy == 'legacy':
        return {**{c: 0.0 for c in graph_cols}, **{c: float(node_fill[c]) for c in agg_cols}}
    if policy == 'row_median':
        med = feat_tr[list(graph_cols) + list(agg_cols)].median()
        return med.fillna(0.0).to_dict()          # tamamen bos kolon -> 0
    raise ValueError(f'IMPUTE_POLICY {policy!r} gecersiz; secenekler: {IMPUTE_POLICIES}')

print(f'A2 imputasyon yardimcisi hazir (IMPUTE_POLICY={IMPUTE_POLICY}).')


A2 imputasyon yardimcisi hazir (IMPUTE_POLICY=row_median).


In [10]:
def sparse_pagerank(rows, cols, n, alpha=PAGERANK_ALPHA, iters=PAGERANK_ITERS, tol=1e-10):
    """Yonsuz (coklu kenar = agirlik) graf icin PageRank; scipy seyrek matris, networkx'ten hizli."""
    if n == 0:
        return np.zeros(0)
    A = sparse.coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(n, n)).tocsr()
    A = A + A.T
    deg = np.asarray(A.sum(axis=1)).ravel()
    inv = np.divide(1.0, deg, out=np.zeros_like(deg), where=deg > 0)
    PT = (sparse.diags(inv) @ A).T.tocsr()
    dangling = deg == 0
    r = np.full(n, 1.0 / n)
    for _ in range(iters):
        r_new = alpha * (PT @ r) + (alpha * r[dangling].sum() + 1.0 - alpha) / n
        done = np.abs(r_new - r).sum() < tol
        r = r_new
        if done:
            break
    return r

# --- adim ici tx-tx yapisi (fold'dan bagimsiz; etiket icermez; tahmin aninda gozlenir) ---
t0 = time.time()
TX_PR_STEP = np.zeros(N_TX); TX_CC_SIZE = np.zeros(N_TX)
_same = TX_EDGES[STEP[TX_EDGES.a.values] == STEP[TX_EDGES.b.values]]
for _s in np.unique(STEP):
    _nodes = np.where(STEP == _s)[0]
    _loc = pd.Series(np.arange(len(_nodes)), index=_nodes)
    _es = _same[STEP[_same.a.values] == _s]
    _r, _c = _loc[_es.a.values].values, _loc[_es.b.values].values
    TX_PR_STEP[_nodes] = sparse_pagerank(_r, _c, len(_nodes)) * len(_nodes)     # adim ici ortalama = 1
    _A = sparse.coo_matrix((np.ones(len(_r)), (_r, _c)), shape=(len(_nodes),) * 2)
    _, _lab = connected_components(_A, directed=False)
    TX_CC_SIZE[_nodes] = np.bincount(_lab)[_lab]
tx['tx_pr_step'], tx['tx_cc_size'] = TX_PR_STEP.astype(np.float32), TX_CC_SIZE.astype(np.float32)
TX_GRAPH = TX_GRAPH_GIVEN + ['tx_pr_step', 'tx_cc_size']
TXF = tx[['txId', 'step', 'y'] + LOCAL_FEATS + AGG_FEATS + TX_STATS + TX_GRAPH]
print(f'Adim ici tx grafi oznitelikleri hazir ({time.time()-t0:.0f}s).')


Adim ici tx grafi oznitelikleri hazir (1s).


## 5. Sızıntısız öznitelik hesaplama (çekirdek)

`compute_fold_features(train, eval, struct)`: cüzdan geçmişi, cüzdan–işlem grafı PageRank'i ve cüzdan fraud oranı
**yalnızca yapı bölümündeki** (`struct`) işlemlerden; fraud oranı yalnızca yapı bölümündeki **etiketli** işlemlerden,
eğitim satırları için iç K-fold ile OOF. Görülmeyen cüzdanlar NaN → tek imputasyon politikası (A2); fraud oranı → önsel.

In [11]:
SIDE_EDGES = {'ent': E_IN, 'cp': E_OUT}          # ent = giris cuzdanlari (varlik), cp = cikis cuzdanlari (karsi taraf)
W_STATS    = ['nsend', 'nrecv', 'nsteps', 'btc', 'pr']

def _wallet_tables(S_mask):
    ein, eout = E_IN[S_mask[E_IN.tx.values]], E_OUT[S_mask[E_OUT.tx.values]]
    n_send = np.bincount(ein.w.values, minlength=N_W).astype(float)
    n_recv = np.bincount(eout.w.values, minlength=N_W).astype(float)
    both = pd.concat([ein, eout], ignore_index=True)
    both['step'], both['btc'] = STEP[both.tx.values], TOTAL_BTC[both.tx.values]
    g = both.groupby('w')
    W = {'nsend': n_send, 'nrecv': n_recv}
    for k, s in [('first', g['step'].min()), ('nsteps', g['step'].nunique()), ('btc', g['btc'].mean())]:
        a = np.full(N_W, np.nan); a[s.index.values] = s.values; W[k] = a
    seen = (n_send + n_recv) > 0
    pr = sparse_pagerank(both.w.values, N_W + both.tx.values, N_W + N_TX)[:N_W]
    W['pr'] = pr * max(1, int(seen.sum()))
    for k in W:
        W[k] = np.where(seen, W[k], np.nan)
    return W, seen, both

def _wallet_rates(both, train_codes, seed):
    """Cuzdan fraud orani. Egitim satirlari: ic K-fold (islem bazli); hem oran hem onsel o fold'un
    etiketleri HARIC hesaplanir -> satir kendi etiketini hicbir yoldan gormez."""
    lab = both[~np.isnan(Y[both.tx.values])][['w', 'tx']].drop_duplicates()
    yv = Y[lab.tx.values]
    s_all = np.bincount(lab.w.values, weights=yv, minlength=N_W)
    c_all = np.bincount(lab.w.values, minlength=N_W).astype(float)
    ltx = np.unique(lab.tx.values)
    prior = float(np.mean(Y[ltx])) if len(ltx) else float(np.nanmean(Y))
    def rate(s, c, p):
        return np.where(c > 0, (s + TE_SMOOTHING * p) / (c + TE_SMOOTHING), np.nan)
    fold_of = np.full(N_TX, -1)
    for k, (_, idx) in enumerate(KFold(TE_N_SPLITS, shuffle=True, random_state=seed).split(train_codes)):
        fold_of[train_codes[idx]] = k
    oof, priors = {}, {}
    for k in range(TE_N_SPLITS):
        m = fold_of[lab.tx.values] == k
        rest = ltx[fold_of[ltx] != k]
        priors[k] = float(np.mean(Y[rest])) if len(rest) else prior
        oof[k] = rate(s_all - np.bincount(lab.w.values[m], weights=yv[m], minlength=N_W),
                      c_all - np.bincount(lab.w.values[m], minlength=N_W), priors[k])
    return rate(s_all, c_all, prior), oof, fold_of, prior, priors

def _side_features(codes, side, W, seen, rate_arr):
    e = SIDE_EDGES[side]; e = e[np.isin(e.tx.values, codes)]
    d = pd.DataFrame({'tx': e.tx.values, 'seen': seen[e.w.values].astype(float), 'first': W['first'][e.w.values],
                      'frate': rate_arr[e.w.values]})
    for k in W_STATS:
        d[k] = W[k][e.w.values]
    g = d.groupby('tx')
    out = pd.DataFrame(index=pd.Index(codes, name='tx'))
    out[f'{side}_seen_frac'] = g['seen'].mean()
    for k in W_STATS:
        out[f'{side}_{k}_mean'] = g[k].mean(); out[f'{side}_{k}_max'] = g[k].max()
    out[f'{side}_age_max'] = STEP[codes] - g['first'].min().reindex(codes).values
    out[f'{side}_frate_mean'] = g['frate'].mean(); out[f'{side}_frate_max'] = g['frate'].max()
    return out

def side_cols(side):
    return {'beh'  : [f'{side}_seen_frac'] + [f'{side}_{k}_{a}' for k in ('nsend', 'nrecv', 'nsteps', 'btc') for a in ('mean', 'max')]
                     + [f'{side}_age_max'],
            'graph': [f'{side}_pr_mean', f'{side}_pr_max'],
            'rate' : [f'{side}_frate_mean', f'{side}_frate_max']}

def compute_fold_features(train_codes, eval_codes, struct_codes, impute_policy=None, verbose=True, te_group_col=None):
    """Tum cuzdan tabanli oznitelikleri YALNIZCA struct_codes islemlerinden ogrenir."""
    t0 = time.time(); timing = {}
    train_codes, eval_codes = np.asarray(train_codes), np.asarray(eval_codes)
    S_mask = np.zeros(N_TX, bool); S_mask[np.asarray(struct_codes)] = True
    assert not S_mask[eval_codes].any(), 'degerlendirme islemleri yapi bolumunde olmamali'
    W, seen, both = _wallet_tables(S_mask)
    full_rate, oof_rate, fold_of, prior, fold_priors = _wallet_rates(both, train_codes, SEED)
    timing['wallet_s'] = time.time() - t0

    def build(codes, is_train):
        f = TXF.iloc[codes].reset_index(drop=True)
        parts = []
        for side in SIDE_EDGES:
            rc = side_cols(side)['rate']
            if is_train:
                blocks = []
                for k in range(TE_N_SPLITS):
                    b = _side_features(codes[fold_of[codes] == k], side, W, seen, oof_rate[k])
                    b[rc] = b[rc].fillna(fold_priors[k])        # gecmissiz cuzdan -> o fold'un onseli
                    blocks.append(b)
                sf = pd.concat(blocks).reindex(codes)
            else:
                sf = _side_features(codes, side, W, seen, full_rate)
                sf[rc] = sf[rc].fillna(prior)
            parts.append(sf.reset_index(drop=True))
        return pd.concat([f] + parts, axis=1)

    f_tr, f_te = build(train_codes, True), build(eval_codes, False)
    for side in SIDE_EDGES:
        for o in (f_tr, f_te):
            o[f'{side}_seen_frac'] = o[f'{side}_seen_frac'].fillna(0.0)
            o[f'_{side}_seen'] = (o[f'{side}_seen_frac'] > 0).astype(np.int8)
    beh   = [c for s in SIDE_EDGES for c in side_cols(s)['beh'] if not c.endswith('seen_frac')]
    graph = [c for s in SIDE_EDGES for c in side_cols(s)['graph']]
    rates = [c for s in SIDE_EDGES for c in side_cols(s)['rate']]
    node_fill = f_tr[beh].median().fillna(0.0).to_dict()
    fills = impute_values(f_tr, graph, beh, node_fill=node_fill, policy=impute_policy)
    fills.update({c: prior for c in rates})
    f_tr, f_te = f_tr.fillna(fills), f_te.fillna(fills)
    f_tr.attrs['cold_values'] = {**fills, **{f'{s}_seen_frac': 0.0 for s in SIDE_EDGES}}
    timing.update({'apply_s': time.time() - t0, 'n_struct_tx': int(S_mask.sum()), 'n_wallets_seen': int(seen.sum()),
                   'kappa': float(f_te['_ent_seen'].mean()), 'kappa_cp': float(f_te['_cp_seen'].mean()),
                   'impute_policy': impute_policy or IMPUTE_POLICY, 'prior': prior})
    if verbose:
        print(f"    yapi={timing['n_struct_tx']:,} islem | gorulen cuzdan={timing['n_wallets_seen']:,} | "
              f"kappa(giris)={timing['kappa']:.3f} kappa(cikis)={timing['kappa_cp']:.3f} | {timing['apply_s']:.1f}s", flush=True)
    return f_tr, f_te, timing

# --- bolmeler ---
def split_temporal(cut, horizon=None):
    ev = (STEP[LAB] > cut) & ((STEP[LAB] <= cut + horizon) if horizon else True)
    return LAB[STEP[LAB] <= cut], LAB[ev], np.where(STEP <= cut)[0]

def splits_random_cv(n_folds=N_FOLDS):
    """P1 (onceki calismalardaki rastgele bolme): yapi = degerlendirme islemleri HARIC tum islemler."""
    out = []
    for tr, te in StratifiedKFold(n_folds, shuffle=True, random_state=SEED).split(LAB, Y[LAB]):
        out.append((LAB[tr], LAB[te], np.setdiff1d(np.arange(N_TX), LAB[te])))
    return out
print('compute_fold_features hazir (Elliptic++: cuzdan gecmisi, cuzdan-islem PageRank, OOF cuzdan fraud orani).')


compute_fold_features hazir (Elliptic++: cuzdan gecmisi, cuzdan-islem PageRank, OOF cuzdan fraud orani).


## 6. Öznitelik setleri

In [12]:
FEATURES_A  = LOCAL_FEATS + TX_STATS                                   # islemin kendi ozellikleri
BEHAVIORAL  = [c for s in SIDE_EDGES for c in side_cols(s)['beh']]      # cuzdan gecmisi (giris + cikis)
GRAPH_FEATS = TX_GRAPH + [c for s in SIDE_EDGES for c in side_cols(s)['graph'] + side_cols(s)['rate']]
FEATURE_SETS = {
    'A (ham)'          : FEATURES_A,
    'B (A+davranışsal)': FEATURES_A + BEHAVIORAL,
    'C (B+graf)'       : FEATURES_A + BEHAVIORAL + GRAPH_FEATS,
    'D (A+graf)'       : FEATURES_A + GRAPH_FEATS,
}
# Elliptic'in hazir 1-hop komsuluk aggregate'leri (Weber vd. 'AF'); Vesta C gibi referans, ablation disi
SET_REF = {'A+AGG (referans)': FEATURES_A + AGG_FEATS}
for k, v in {**FEATURE_SETS, **SET_REF}.items():
    print(f'  {k:20s}: {len(v):3d} oznitelik')
print(f'  davranissal={len(BEHAVIORAL)} | graf={len(GRAPH_FEATS)} (tx grafi {len(TX_GRAPH)}, cuzdan PageRank + fraud orani)')


  A (ham)             : 108 oznitelik
  B (A+davranışsal)   : 128 oznitelik
  C (B+graf)          : 140 oznitelik
  D (A+graf)          : 120 oznitelik
  A+AGG (referans)    : 180 oznitelik
  davranissal=20 | graf=12 (tx grafi 4, cuzdan PageRank + fraud orani)


## 7. Modeller ve metrikler (diğer notebook'larla aynı)

In [13]:
def get_models(fast=FAST_MODE):
    '''10 siniflandirici.

    NOT 1: 'Gradient Boosted Decision Trees' (onceki surumde yanlislikla
           'Gradient Augmented' yazilmisti).
    NOT 2: GBDT icin HistGradientBoostingClassifier kullanilir; sklearn'un
           histogram tabanli, cok cekirdekli gradient boosting uygulamasi.
           Klasik GradientBoostingClassifier ile ayni algoritma ailesi,
           bu veri boyutunda ~300x daha hizli.
    '''
    n_est = 50 if fast else 100
    return {
        'LR'       : LogisticRegression(max_iter=500, random_state=SEED),
        'NB'       : GaussianNB(),
        'DT'       : DecisionTreeClassifier(max_depth=15, random_state=SEED),
        'KNN'      : KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        'SVM'      : CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=3000, dual='auto',
                                                      random_state=SEED), cv=3),
        'RF'       : RandomForestClassifier(n_estimators=n_est, max_depth=12,
                                            random_state=SEED, n_jobs=-1),
        'GBDT'     : HistGradientBoostingClassifier(max_iter=n_est, max_depth=6, random_state=SEED),
        'XGBoost'  : XGBClassifier(n_estimators=n_est, max_depth=6, random_state=SEED,
                                   n_jobs=-1, verbosity=0, eval_metric='logloss'),
        'LightGBM' : LGBMClassifier(n_estimators=n_est, random_state=SEED, n_jobs=-1, verbose=-1),
        'MLP'      : MLPClassifier(hidden_layer_sizes=(64,32), max_iter=60,
                                   early_stopping=True, random_state=SEED),
    }

HYPERPARAMS = {
    'LR'      : 'max_iter=500, C=1.0, solver=lbfgs',
    'NB'      : 'GaussianNB varsayilan',
    'DT'      : 'max_depth=15, criterion=gini',
    'KNN'     : f'n_neighbors=5, metric=minkowski(p=2), egitim seti {KNN_TRAIN_CAP:,} ile sinirli',
    'SVM'     : 'LinearSVC(C=1.0, max_iter=3000) + CalibratedClassifierCV(cv=3)',
    'RF'      : f'n_estimators={50 if FAST_MODE else 100}, max_depth=12, bootstrap=True',
    'GBDT'    : f'HistGradientBoosting, max_iter={50 if FAST_MODE else 100}, max_depth=6, lr=0.1',
    'XGBoost' : f'n_estimators={50 if FAST_MODE else 100}, max_depth=6, lr=0.3',
    'LightGBM': f'n_estimators={50 if FAST_MODE else 100}, num_leaves=31, lr=0.1',
    'MLP'     : 'hidden_layer_sizes=(64,32), max_iter=60, early_stopping=True, adam',
}


def full_metrics(y_true, y_pred, y_score, k_frac=0.01):
    '''Dengesiz siniflandirma icin kapsamli metrik seti .'''
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    n_k = max(1, int(len(y_true) * k_frac))
    top = np.argsort(-y_score)[:n_k]
    return {'accuracy':accuracy_score(y_true,y_pred),
            'precision':precision_score(y_true,y_pred,zero_division=0),
            'recall':recall_score(y_true,y_pred,zero_division=0),
            'f1':f1_score(y_true,y_pred,zero_division=0),
            'auc_roc':roc_auc_score(y_true,y_score),
            'pr_auc':average_precision_score(y_true,y_score),
            'mcc':matthews_corrcoef(y_true,y_pred),
            'fpr': fp/(fp+tn) if (fp+tn) else 0.0,
            'precision_at_1pct': y_true[top].sum()/n_k,
            'recall_at_1pct': y_true[top].sum()/max(1, y_true.sum()),
            'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}


THRESHOLD_MODE = 'tuned'   # 'tuned' | 'fixed'
VAL_FRAC = 0.20            # eşik seçimi için eğitim fold'undan ayrılan pay

def _pick_threshold(y_val, s_val):
    """Eğitim fold'unun validasyon kısmında F1'i maksimize eden eşik."""
    prec, rec, thr = precision_recall_curve(y_val, s_val)
    f1 = 2*prec*rec / (prec + rec + 1e-12)
    if len(thr) == 0:
        return 0.5
    return float(thr[max(0, np.nanargmax(f1[:-1]))])


def fit_predict(model, Xtr, ytr, Xte, sampler=None, model_name=None):
    """Ölçekleme, örnekleme ve eşik seçimi YALNIZCA eğitim verisinde yapılır."""
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr); Xte_s = sc.transform(Xte)
    if sampler is not None:
        Xtr_s, ytr = sampler.fit_resample(Xtr_s, ytr)
    if model_name == 'KNN' and len(Xtr_s) > KNN_TRAIN_CAP:
        rs = np.random.RandomState(SEED)
        pos = np.where(ytr==1)[0]; neg = np.where(ytr==0)[0]
        n_neg = max(1, KNN_TRAIN_CAP - len(pos))
        keep = np.concatenate([pos, rs.choice(neg, min(n_neg, len(neg)), replace=False)])
        rs.shuffle(keep); Xtr_s, ytr = Xtr_s[keep], ytr[keep]

    thr = 0.5
    t0 = time.time()
    if THRESHOLD_MODE == 'tuned':
        # eğitim fold'unu ikiye ayır: alt-eğitim + validasyon (eşik için)
        from sklearn.model_selection import train_test_split
        Xa, Xv, ya, yv = train_test_split(Xtr_s, ytr, test_size=VAL_FRAC,
                                          stratify=ytr, random_state=SEED)
        model.fit(Xa, ya)
        try:    sv = model.predict_proba(Xv)[:,1]
        except Exception: sv = model.decision_function(Xv)
        thr = _pick_threshold(yv, sv)
        # eşik belirlendikten sonra TÜM eğitim fold'u ile yeniden eğit
        model.fit(Xtr_s, ytr)
    else:
        model.fit(Xtr_s, ytr)
    fit_s = time.time() - t0

    try:    ys = model.predict_proba(Xte_s)[:,1]
    except Exception: ys = model.decision_function(Xte_s)
    yp = (ys >= thr).astype(int)
    return yp, ys, fit_s, thr

print('Modeller ve metrikler hazır.')

Modeller ve metrikler hazır.


## 8. Artefakt denetimi (A6 kriterleri)

In [14]:
# ============================================================
# A6 — ON-DENEY ARTEFAKT DENETIMI (makale Table 2 ile ayni tanimlar)
# ------------------------------------------------------------
# C1  a*    : her tek oznitelik icin AUC (sayisal: ham deger, kategorik: OOF oran)
# C2  F1*   : tek kosullu (x>t, x<=t, x==v) ve iki kosullu (AND / OR) kurallar
# C3  rho1  : tek islemli varlik orani
# C4  lambda: kenar / dugum orani      C5 gamma: en buyuk bilesenin dugum payi
# Esikler analizden ONCE sabitlenmistir; burada yalnizca parametre olarak tutulur.
# ============================================================
AUDIT_THRESHOLDS = {
    'C1  En iyi tek-oznitelik AUC (a*)':   ('<', 0.90),
    'C2  En iyi basit kural F1 (F1*)':     ('<', 0.50),
    'C3  Tek islemli varlik orani (rho1)': ('<', 0.50),
    'C4  Kenar/dugum orani (lambda)':      ('>', 1.5),
    'C5  En buyuk bilesen payi (gamma)':   ('>', 0.80),
}
AUDIT_QUANTILES  = np.linspace(0.01, 0.99, 99)   # sayisal kural esikleri
AUDIT_TOPK       = 5                             # iki kosullu taramaya giren, oznitelik basina en iyi tek kural
AUDIT_MAX_LEVELS = 200                           # kategorik: en sik N seviye
AUDIT_KEEP_RULES = 200                           # raporlanan kural sayisi

def _rule_stats(y, m):
    tp = np.count_nonzero(y & m); p = np.count_nonzero(m); P = np.count_nonzero(y)
    return (tp / p if p else 0.0), (tp / P if P else 0.0), (2.0 * tp / (p + P) if (p + P) else 0.0)

def audit_single_feature_auc(d, y, num_cols, cat_cols):
    rows = []
    for c in num_cols:
        v = d[c].astype(float); v = v.fillna(v.median()).values
        if np.std(v) == 0: continue
        a = roc_auc_score(y, v); rows.append({'feature': c, 'kind': 'sayisal', 'auc': max(a, 1 - a)})
    for c in cat_cols:
        codes = pd.factorize(d[c].astype(str))[0].astype(float)
        s, _ = oof_target_encode(codes, y, codes[:1])          # kategori orani, OOF (kendi etiketi yok)
        if np.std(s) == 0: continue
        a = roc_auc_score(y, s); rows.append({'feature': c, 'kind': 'kategorik (OOF oran)', 'auc': max(a, 1 - a)})
    return pd.DataFrame(rows).sort_values('auc', ascending=False).reset_index(drop=True)

def audit_rules(d, y, num_cols, cat_cols, topk=AUDIT_TOPK):
    y = np.asarray(y).astype(bool)
    singles = []                                               # (f1, ad, oznitelik, maske)
    for c in num_cols:
        x = d[c].astype(float).values; ok = ~np.isnan(x); cand = []
        for t in np.unique(np.nanquantile(x, AUDIT_QUANTILES)):
            for op, m in (('>', ok & (x > t)), ('<=', ok & (x <= t))):
                cand.append((_rule_stats(y, m)[2], f'{c} {op} {t:.4g}', c, m))
        cand.sort(key=lambda r: -r[0]); singles += cand[:topk]
    for c in cat_cols:
        v = d[c].astype(str).values
        cand = [(_rule_stats(y, v == L)[2], f'{c} == {L}', c, v == L)
                for L in pd.Series(v).value_counts().index[:AUDIT_MAX_LEVELS]]
        cand.sort(key=lambda r: -r[0]); singles += cand[:topk]
    rows = []
    for f, n, c, m in singles:
        p, r, f1 = _rule_stats(y, m); rows.append((n, 1, p, r, f1))
    for i in range(len(singles)):
        for j in range(i + 1, len(singles)):
            if singles[i][2] == singles[j][2]: continue        # iki kosul farkli ozniteliklerden
            for op, m in (('AND', singles[i][3] & singles[j][3]), ('OR', singles[i][3] | singles[j][3])):
                p, r, f1 = _rule_stats(y, m)
                rows.append((f'({singles[i][1]}) {op} ({singles[j][1]})', 2, p, r, f1))
    out = pd.DataFrame(rows, columns=['Kural', 'Kosul', 'Precision', 'Recall', 'F1'])
    return out.sort_values('F1', ascending=False).head(AUDIT_KEEP_RULES).reset_index(drop=True)

def audit_topology(entity, edges):
    """entity: varlik kimligi serisi; edges: DataFrame(u, v) tekil kenarlar."""
    cnt = entity.value_counts()
    G = nx.from_pandas_edgelist(edges, 'u', 'v')
    n = max(1, G.number_of_nodes())
    lcc = max(nx.connected_components(G), key=len) if G.number_of_nodes() else set()
    return float((cnt == 1).mean()), G.number_of_edges() / n, len(lcc) / n

def artefact_audit(d, label, num_cols, cat_cols, entity_col, edges, thresholds=AUDIT_THRESHOLDS):
    y = d[label].values.astype(int)
    single = audit_single_feature_auc(d, y, num_cols, cat_cols)
    rules  = audit_rules(d, y, num_cols, cat_cols)
    rho1, lam, gamma = audit_topology(d[entity_col], edges)
    vals = [single['auc'].iloc[0], rules['F1'].iloc[0], rho1, lam, gamma]
    rows = []
    for (name, (op, thr)), v in zip(thresholds.items(), vals):
        ok = (v < thr) if op == '<' else (v > thr)
        rows.append({'Kriter': name, 'Kabul': f'{op} {thr}', 'Deger': round(float(v), 4),
                     'Sonuc': 'GECTI' if ok else '!! KALDI'})
    return pd.DataFrame(rows), single, rules

print('A6 artefakt denetimi yardimcilari hazir.')


A6 artefakt denetimi yardimcilari hazir.


In [15]:
RUN_AUDIT = True

if RUN_AUDIT:
    _tr, _ev, _S = split_temporal(TEMPORAL_CUT)
    _d = TXF.iloc[_tr]; _y = _d['y'].values.astype(int)
    scan  = audit_single_feature_auc(_d, _y, FEATURES_A, [])
    rule_df = audit_rules(_d, _y, FEATURES_A, [])
    _Sm = np.zeros(N_TX, bool); _Sm[_S] = True
    _ein, _eout = E_IN[_Sm[E_IN.tx.values]], E_OUT[_Sm[E_OUT.tx.values]]
    _ns = np.bincount(_ein.w.values, minlength=N_W)
    _rho1 = float((_ns == 1).sum() / max(1, (_ns > 0).sum()))         # tek islemle gonderen giris cuzdani orani
    _both = pd.concat([_ein, _eout])
    _wn = np.unique(_both.w.values); _tn = np.unique(_both.tx.values)
    _n = len(_wn) + len(_tn)
    _r = np.searchsorted(_wn, _both.w.values); _c = len(_wn) + np.searchsorted(_tn, _both.tx.values)
    _, _lab = connected_components(sparse.coo_matrix((np.ones(len(_r)), (_r, _c)), shape=(_n, _n)), directed=False)
    _lam, _gam = len(_both.drop_duplicates()) / _n, np.bincount(_lab).max() / _n
    _vals = [scan['auc'].iloc[0], rule_df['F1'].iloc[0], _rho1, _lam, _gam]
    audit_table = pd.DataFrame([{'Kriter': k, 'Kabul': f'{op} {t}', 'Deger': round(float(v), 4),
                                 'Sonuc': 'GECTI' if ((v < t) if op == '<' else (v > t)) else '!! KALDI'}
                                for (k, (op, t)), v in zip(AUDIT_THRESHOLDS.items(), _vals)])
    print('=' * 88); print('ARTEFAKT DENETIMI — Elliptic++ (egitim: adim 1-34; graf: cuzdan-islem)'); print('=' * 88)
    print(audit_table.to_string(index=False)); print()
    print('En yuksek tek-oznitelik AUC (ilk 10):'); print(scan.head(10).round(4).to_string(index=False)); print()
    print('En iyi kurallar (ilk 5):'); print(rule_df.head(5).round(4).to_string(index=False))
    audit_table.to_csv('elliptic_artefact_audit.csv', index=False)


ARTEFAKT DENETIMI — Elliptic++ (egitim: adim 1-34; graf: cuzdan-islem)
                             Kriter Kabul  Deger    Sonuc
  C1  En iyi tek-oznitelik AUC (a*) < 0.9 0.8663    GECTI
    C2  En iyi basit kural F1 (F1*) < 0.5 0.7744 !! KALDI
C3  Tek islemli varlik orani (rho1) < 0.5 0.9329 !! KALDI
     C4  Kenar/dugum orani (lambda) > 1.5 1.2643 !! KALDI
  C5  En buyuk bilesen payi (gamma) > 0.8 1.0000    GECTI

En yuksek tek-oznitelik AUC (ilk 10):
             feature    kind    auc
                size sayisal 0.8663
    Local_feature_53 sayisal 0.8548
    Local_feature_55 sayisal 0.8364
    Local_feature_90 sayisal 0.8320
num_output_addresses sayisal 0.8123
     Local_feature_5 sayisal 0.8114
    Local_feature_14 sayisal 0.8113
    Local_feature_29 sayisal 0.8085
    Local_feature_23 sayisal 0.8085
    Local_feature_60 sayisal 0.8046

En iyi kurallar (ilk 5):
                                                          Kural  Kosul  Precision  Recall     F1
(Local_feature_55 <= -0

## 9. Sızıntı öz-testi

In [16]:
RUN_LEAK_SELFTEST = True

if RUN_LEAK_SELFTEST:
    _tr, _ev, _ = split_temporal(TEMPORAL_CUT)
    _Sa = np.where((STEP <= TEMPORAL_CUT) & (STEP % 2 == 1))[0]; _Sb = np.where((STEP <= TEMPORAL_CUT) & (STEP % 2 == 0))[0]
    _, _fa, _ = compute_fold_features(np.intersect1d(_tr, _Sa), _ev, _Sa, verbose=False)
    _, _fb, _ = compute_fold_features(np.intersect1d(_tr, _Sb), _ev, _Sb, verbose=False)
    _rows = []
    for c in ['ent_nsend_max', 'ent_pr_max', 'ent_frate_max', 'cp_pr_max', 'cp_frate_max', 'tx_pr_step']:
        _same = bool(np.allclose(_fa[c].values, _fb[c].values))
        _rows.append({'oznitelik': c, 'ayni_mi': _same, 'beklenen': 'ayni (adim ici yapi, fold\'dan bagimsiz)' if c.startswith('tx_') else 'farkli'})
    leak_selftest = pd.DataFrame(_rows); print(leak_selftest.to_string(index=False))
    _bad = leak_selftest[(leak_selftest.ayni_mi) & (leak_selftest.beklenen == 'farkli')]
    print('SONUC:', 'cuzdan oznitelikleri yapi bolumune bagli' if _bad.empty else f'!! bagimsiz hesaplanan: {_bad.oznitelik.tolist()}')


    oznitelik  ayni_mi                                beklenen
ent_nsend_max    False                                  farkli
   ent_pr_max    False                                  farkli
ent_frate_max    False                                  farkli
    cp_pr_max    False                                  farkli
 cp_frate_max    False                                  farkli
   tx_pr_step     True ayni (adim ici yapi, fold'dan bagimsiz)
SONUC: cuzdan oznitelikleri yapi bolumune bagli


## 10. Hızlı kontrol — standart zamansal bölme (1–34 / 35–49)
Tam grid E2'de. Burada birkaç modelle setlerin makul çalıştığı ve κ'ya göre ayrılmış PR-AUC görülür.
Referans büyüklük: Weber vd. (2019) RF + tüm öznitelikler (AF) için illicit F1 ≈ 0.79 (eşik 0.5; buradaki eşik eğitimde seçiliyor).

In [17]:
RUN_E1_CHECK = False   # E2 tam grid bunun yerini aliyor
E1_MODELS    = ['LR', 'RF', 'LightGBM']

if RUN_E1_CHECK:
    _tr, _ev, _S = split_temporal(TEMPORAL_CUT)
    f_tr, f_te, tinfo = compute_fold_features(_tr, _ev, _S)
    _ytr, _yte = f_tr['y'].values.astype(int), f_te['y'].values.astype(int)
    _seen = f_te['_ent_seen'].values.astype(bool)
    _rows = []
    for _s, _cols in {**FEATURE_SETS, **SET_REF}.items():
        _Xtr = np.nan_to_num(f_tr[_cols].astype(np.float64).values); _Xte = np.nan_to_num(f_te[_cols].astype(np.float64).values)
        for _m in E1_MODELS:
            _yp, _ys, _fs, _thr = fit_predict(get_models()[_m], _Xtr, _ytr, _Xte, model_name=_m)
            _r = full_metrics(_yte, _yp, _ys)
            _sub = {f'pr_auc_{n}': (average_precision_score(_yte[mk], _ys[mk]) if mk.sum() and len(np.unique(_yte[mk])) > 1 else np.nan)
                    for n, mk in [('gorulen', _seen), ('gorulmeyen', ~_seen)]}
            _rows.append({'set': _s, 'model': _m, 'f1': _r['f1'], 'pr_auc': _r['pr_auc'], 'precision': _r['precision'],
                          'recall': _r['recall'], **_sub, 'threshold': _thr})
    e1_check = pd.DataFrame(_rows)
    e1_check.to_csv(f'elliptic_e1_check_{PIPELINE_VERSION}.csv', index=False)
    print(f"kappa(giris cuzdani) = {tinfo['kappa']:.3f} | test n = {len(_yte):,} (illicit {int(_yte.sum()):,})")
    for _met in ['f1', 'pr_auc', 'pr_auc_gorulen', 'pr_auc_gorulmeyen']:
        if e1_check[_met].notna().sum() == 0:
            print(f'{_met}: bu alt grupta iki sinifli ornek yok, atlandi.'); continue
        _p = e1_check.pivot_table(index='model', columns='set', values=_met, dropna=False).reindex(columns=[*FEATURE_SETS, *SET_REF])
        _p['D-A'] = _p['D (A+graf)'] - _p['A (ham)']; _p['C-B'] = _p['C (B+graf)'] - _p['B (A+davranışsal)']
        print('=' * 110); print(f'E1 hizli kontrol — {_met}'); print('=' * 110); print(_p.round(4).to_string()); print()


## 11. E2 — Protokoller ve çalıştırıcı
Her fold için `compute_fold_features` bir kez çağrılır; tüm set × model kombinasyonları o fold'un özniteliklerini kullanır.
Sonuçlar fold bitiminde `CHECKPOINT_DIR`'e yazılır (`RESUME=True` ile kaldığı yerden devam).

In [18]:
E2_MODELS       = ['LR', 'RF', 'XGBoost', 'LightGBM']   # MLP icin listeye ekleyin (fold basina ~1 dk daha)
E2_SETS         = {**FEATURE_SETS, **SET_REF}
KAPPA_MASK_SETS = ['C (B+graf)', 'D (A+graf)']
ROUTER_PAIRS    = [('C (B+graf)', 'A (ham)'), ('D (A+graf)', 'A (ham)')]
KAPPA_SIDES     = {'ent': '_ent_seen', 'cp': '_cp_seen'}       # taraf -> degerlendirme satirinda 'goruldu' bayragi
ROUTER_SEEN_COL = '_ent_seen'                                  # router: giris cuzdani gorulduyse graf-kosullu model

PROTOCOLS = {
    'P1': ('Random 5-fold CV', splits_random_cv()),
    'P3': ('Temporal 1-34 / 35-49', [split_temporal(TEMPORAL_CUT)]),
    'P4': ('Rolling-origin', [split_temporal(o, ROLLING_HORIZON) for o in ROLLING_ORIGINS]),
}
FOLD_LABEL = {'P4': {k + 1: f'origin {o}' for k, o in enumerate(ROLLING_ORIGINS)}}
for _k, (_lab, _spl) in PROTOCOLS.items():
    print(f'{_k} {_lab:24s}: {len(_spl)} fold | egitim {[len(s[0]) for s in _spl]} | degerlendirme {[len(s[1]) for s in _spl]}')

def side_entity_cols(side):
    return side_cols(side)['beh'] + side_cols(side)['graph'] + side_cols(side)['rate']

def kappa_mask_two_sided(f_tr, f_te, seed):
    """Her taraf icin egitim islemlerinin (1 - kappa_hat_taraf) kadarini cold-start gibi gosterir."""
    cv = f_tr.attrs['cold_values']; rs = np.random.RandomState(seed)
    tr, te, info = f_tr.copy(), f_te.copy(), {}
    for side, scol in KAPPA_SIDES.items():
        share = 1.0 - float(f_te[scol].mean())
        m = rs.rand(len(tr)) < share
        cols = [c for c in side_entity_cols(side) if c in tr.columns and c in cv]
        tr[cols] = tr[cols].astype(np.float64)
        if m.any():
            for c in cols:
                tr.loc[m, c] = cv[c]
        tr[f'is_cold_{side}'] = (m | (tr[scol].values == 0)).astype(float)
        te[f'is_cold_{side}'] = (te[scol].values == 0).astype(float)
        info[f'mask_share_{side}'] = share; info[f'masked_frac_{side}'] = float(m.mean())
    return tr, te, info

def _X(frame, cols):
    return np.nan_to_num(frame[cols].astype(np.float64).values, nan=0.0, posinf=0.0, neginf=0.0)

def run_protocol_e(key, model_names=None, feature_sets=None, kappa_mask_sets=KAPPA_MASK_SETS,
                   router_pairs=ROUTER_PAIRS, feat_kwargs=None, tag=None, verbose=True):
    label, splits = PROTOCOLS[key]
    model_names = model_names or E2_MODELS; feature_sets = feature_sets or E2_SETS
    tag = tag or f'E2_{key}'
    rp, tp = (os.path.join(CHECKPOINT_DIR, f'{tag}_{x}.csv') for x in ('results', 'timings'))
    rows, timings, done = [], [], set()
    if RESUME and os.path.exists(rp):
        prev = pd.read_csv(rp); rows = prev.to_dict('records'); done = set(prev['fold'].astype(int))
        if os.path.exists(tp): timings = pd.read_csv(tp).to_dict('records')
        print(f'  [DEVAM] {tag}: tamamlanmis fold = {sorted(done)}', flush=True)
    for fold, (tr_c, ev_c, st_c) in enumerate(splits, start=1):
        if fold in done:
            continue
        t_fold = time.time()
        f_tr, f_te, tinfo = compute_fold_features(tr_c, ev_c, st_c, verbose=verbose, **(feat_kwargs or {}))
        tinfo['fold'] = fold; timings.append(tinfo)
        y_tr, y_te = f_tr['y'].values.astype(int), f_te['y'].values.astype(int)
        out, base = {}, {'protocol': label, 'fold': fold, 'fold_label': FOLD_LABEL.get(key, {}).get(fold, str(fold)),
                         'kappa_ent': tinfo['kappa'], 'kappa_cp': tinfo['kappa_cp'], 'n_eval': len(y_te),
                         'n_pos_eval': int(y_te.sum())}
        def _fit(sname, Xtr, Xte, extra=None):
            for m in model_names:
                try:
                    yp, ys, fit_s, thr = fit_predict(get_models()[m], Xtr, y_tr, Xte, model_name=m)
                    r = full_metrics(y_te, yp, ys); r.update({**base, 'feature_set': sname, 'model': m,
                                                              'fit_s': fit_s, 'threshold': thr, **(extra or {})})
                    rows.append(r); out[(sname, m)] = (ys, yp)
                except Exception as e:
                    print(f'      HATA {sname}/{m}: {e}', flush=True)
        for sname, cols in feature_sets.items():
            _fit(sname, _X(f_tr, cols), _X(f_te, cols))
        if kappa_mask_sets:
            mtr, mte, minfo = kappa_mask_two_sided(f_tr, f_te, SEED + fold)
            extra_cols = [f'is_cold_{s}' for s in KAPPA_SIDES]
            for b in kappa_mask_sets:
                cols = feature_sets[b] + extra_cols
                _fit(f'{b} [κ-mask]', _X(mtr, cols), _X(mte, cols), minfo)
            del mtr, mte
        seen = f_te[ROUTER_SEEN_COL].values.astype(bool)
        for warm, cold in (router_pairs or []):
            for m in model_names:
                if (warm, m) in out and (cold, m) in out:
                    ys = np.where(seen, out[(warm, m)][0], out[(cold, m)][0])
                    yp = np.where(seen, out[(warm, m)][1], out[(cold, m)][1]).astype(int)
                    r = full_metrics(y_te, yp, ys); r.update({**base, 'feature_set': f'Router {warm} | {cold}', 'model': m})
                    rows.append(r); out[(f'Router {warm} | {cold}', m)] = (ys, yp)
        keys = list(out)
        np.savez_compressed(os.path.join(CHECKPOINT_DIR, f'{tag}_fold{fold}_preds.npz'),
                            names=np.array([f'{a}||{b}' for a, b in keys]),
                            scores=np.vstack([out[k][0] for k in keys]).astype(np.float32),
                            preds=np.vstack([out[k][1] for k in keys]).astype(np.int8),
                            y=y_te.astype(np.int8), seen_ent=f_te['_ent_seen'].values.astype(np.int8),
                            seen_cp=f_te['_cp_seen'].values.astype(np.int8), step=f_te['step'].values.astype(np.int16))
        pd.DataFrame(rows).to_csv(rp, index=False); pd.DataFrame(timings).to_csv(tp, index=False)
        if verbose:
            print(f'  [{label}] fold {fold}/{len(splits)} tamam ({time.time()-t_fold:.0f}s)', flush=True)
        del f_tr, f_te; gc.collect()
    return pd.DataFrame(rows), pd.DataFrame(timings)
print('E2 calistirici hazir.')


P1 Random 5-fold CV        : 5 fold | egitim [37251, 37251, 37251, 37251, 37252] | degerlendirme [9313, 9313, 9313, 9313, 9312]
P3 Temporal 1-34 / 35-49   : 1 fold | egitim [29894] | degerlendirme [16670]
P4 Rolling-origin          : 5 fold | egitim [23606, 26381, 29894, 35380, 42838] | degerlendirme [2775, 3513, 5486, 7458, 3726]
E2 calistirici hazir.


## 12. E2 — Tam grid

In [19]:
RUN_E2 = {'P1': True, 'P3': True, 'P4': True}

e2_res = {}
for _k, _on in RUN_E2.items():
    if _on:
        print(f'=== {_k}: {PROTOCOLS[_k][0]}', flush=True)
        e2_res[_k], _ = run_protocol_e(_k)
E2 = pd.concat(e2_res.values(), ignore_index=True)
E2.to_csv(f'elliptic_e2_results_{PIPELINE_VERSION}.csv', index=False)

SHORT = {'A (ham)': 'A', 'B (A+davranışsal)': 'B', 'C (B+graf)': 'C', 'D (A+graf)': 'D', 'A+AGG (referans)': 'A+AGG',
         'C (B+graf) [κ-mask]': 'C κ-mask', 'D (A+graf) [κ-mask]': 'D κ-mask',
         'Router C (B+graf) | A (ham)': 'Router C|A', 'Router D (A+graf) | A (ham)': 'Router D|A'}

def e2_table(res, met):
    p = res.pivot_table(index=['protocol', 'model'], columns='feature_set', values=met)
    o = pd.DataFrame(index=p.index)
    o['kappa_ent'] = res.groupby(['protocol', 'model'])['kappa_ent'].mean()
    for full, short in SHORT.items():
        if full in p: o[short] = p[full]
    o['D−A'] = o['D'] - o['A']; o['C−B'] = o['C'] - o['B']
    o['Cκ−C'] = o['C κ-mask'] - o['C']; o['RouterC−A'] = o['Router C|A'] - o['A']; o['RouterC−C'] = o['Router C|A'] - o['C']
    mean = o.groupby(level='protocol').mean(); mean.index = pd.MultiIndex.from_tuples([(i, 'ORTALAMA') for i in mean.index])
    order = list(dict.fromkeys(res['protocol']))
    o = pd.concat([o, mean]); return o.loc[[ix for pr in order for ix in o.index if ix[0] == pr]].round(4)

e2_tables = {m: e2_table(E2, m) for m in ['f1', 'pr_auc']}
for _m, _t in e2_tables.items():
    print('=' * 140); print(f'E2 — {_m}'); print('=' * 140); print(_t.to_string()); print()


=== P1: Random 5-fold CV
  [DEVAM] E2_P1: tamamlanmis fold = [1, 2, 3, 4, 5]
=== P3: Temporal 1-34 / 35-49
  [DEVAM] E2_P3: tamamlanmis fold = [1]
=== P4: Rolling-origin
  [DEVAM] E2_P4: tamamlanmis fold = [1, 2, 3, 4, 5]
E2 — f1
                                kappa_ent       A       B       C       D   A+AGG  C κ-mask  D κ-mask  Router C|A  Router D|A     D−A     C−B    Cκ−C  RouterC−A  RouterC−C
Random 5-fold CV      LR           0.8094  0.7959  0.5359  0.8763  0.9052  0.7805    0.8745    0.8693      0.8690      0.8799  0.1093  0.3405 -0.0019     0.0732    -0.0073
                      LightGBM     0.8094  0.9342  0.9315  0.9646  0.9650  0.9474    0.9624    0.9648      0.9573      0.9575  0.0308  0.0330 -0.0021     0.0230    -0.0073
                      RF           0.8094  0.9141  0.9148  0.9517  0.9564  0.9393    0.9543    0.9552      0.9424      0.9451  0.0423  0.0369  0.0025     0.0283    -0.0093
                      XGBoost      0.8094  0.9367  0.9368  0.9653  0.9650  0.9513 

### Rolling-origin: origin bazında κ ve graf katkısı (drift)

In [20]:
if 'P4' in e2_res:
    _r = e2_res['P4']
    for _m in ['pr_auc', 'f1']:
        _p = _r.pivot_table(index='fold_label', columns='feature_set', values=_m, aggfunc='mean')
        _o = pd.DataFrame({'kappa_ent': _r.groupby('fold_label')['kappa_ent'].first(),
                           'n_pos': _r.groupby('fold_label')['n_pos_eval'].first(),
                           'A': _p['A (ham)'], 'C': _p['C (B+graf)'], 'D': _p['D (A+graf)'],
                           'C κ-mask': _p['C (B+graf) [κ-mask]'], 'Router C|A': _p['Router C (B+graf) | A (ham)'],
                           'A+AGG': _p['A+AGG (referans)']})
        _o['D−A'] = _o['D'] - _o['A']; _o['RouterC−A'] = _o['Router C|A'] - _o['A']; _o['Cκ−C'] = _o['C κ-mask'] - _o['C']
        _o = _o.reindex([f'origin {o}' for o in ROLLING_ORIGINS])
        print('=' * 110); print(f'Rolling-origin — {_m} (modellerin ortalamasi)'); print('=' * 110); print(_o.round(4).to_string()); print()


Rolling-origin — pr_auc (modellerin ortalamasi)
            kappa_ent  n_pos       A       C       D  C κ-mask  Router C|A   A+AGG     D−A  RouterC−A    Cκ−C
fold_label                                                                                                   
origin 24      0.4148    652  0.9079  0.9493  0.9571    0.9640      0.9696  0.9263  0.0492     0.0617  0.0147
origin 29      0.3518    591  0.8879  0.9184  0.9320    0.9374      0.9526  0.8888  0.0441     0.0646  0.0190
origin 34      0.3963    447  0.8154  0.8408  0.8559    0.8605      0.8746  0.8432  0.0404     0.0592  0.0197
origin 39      0.2482    515  0.7193  0.6666  0.6799    0.6852      0.7284  0.7622 -0.0394     0.0091  0.0186
origin 44      0.4796    121  0.1168  0.0847  0.0945    0.1175      0.1360  0.2691 -0.0223     0.0192  0.0328

Rolling-origin — f1 (modellerin ortalamasi)
            kappa_ent  n_pos       A       C       D  C κ-mask  Router C|A   A+AGG     D−A  RouterC−A    Cκ−C
fold_label                 

### κ'ya göre ayrılmış metrikler (alt grup prevalansıyla)

In [21]:
import glob
RUN_E2_STRAT = True
if RUN_E2_STRAT:
    _rows = []
    for _f in sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'E2_P*_fold*_preds.npz'))):
        _z = np.load(_f); _tag = os.path.basename(_f).split('_fold')[0]
        _fold = int(os.path.basename(_f).split('_fold')[1].split('_')[0])
        _y, _se = _z['y'].astype(int), _z['seen_ent'].astype(bool)
        for _k, _name in enumerate(_z['names']):
            _set, _model = str(_name).split('||')
            for _sub, _mk in [('gorulen', _se), ('gorulmeyen', ~_se)]:
                if _mk.sum() == 0 or len(np.unique(_y[_mk])) < 2:
                    continue
                _rows.append({'tag': _tag, 'fold': _fold, 'subset': _sub, 'n': int(_mk.sum()), 'n_pos': int(_y[_mk].sum()),
                              'prevalans': float(_y[_mk].mean()), 'feature_set': _set, 'model': _model,
                              'pr_auc': average_precision_score(_y[_mk], _z['scores'][_k][_mk]),
                              'f1': f1_score(_y[_mk], _z['preds'][_k][_mk], zero_division=0)})
    e2_strat = pd.DataFrame(_rows)
    e2_strat.to_csv(f'elliptic_e2_kappa_stratified_{PIPELINE_VERSION}.csv', index=False)
    _sz = e2_strat.groupby(['tag', 'subset'])[['n', 'n_pos', 'prevalans']].mean().round(4)
    print('Alt grup boyutlari (fold ortalamasi):'); print(_sz.to_string()); print()
    for _m in ['pr_auc', 'f1']:
        _p = e2_strat.pivot_table(index=['tag', 'subset', 'feature_set'], columns='model', values=_m)
        _p['ortalama'] = _p.mean(axis=1)
        print('=' * 110); print(f'E2 — kappa-ayrilmis {_m}'); print('=' * 110); print(_p.round(4).to_string()); print()


Alt grup boyutlari (fold ortalamasi):
                        n   n_pos  prevalans
tag   subset                                
E2_P1 gorulen      7538.0   669.8     0.0888
      gorulmeyen   1774.8   239.2     0.1346
E2_P3 gorulen      5031.0     5.0     0.0010
      gorulmeyen  11639.0  1078.0     0.0926
E2_P4 gorulen      1639.8    72.2     0.0602
      gorulmeyen   2951.8   393.0     0.1513

E2 — kappa-ayrilmis pr_auc
model                                             LR  LightGBM      RF  XGBoost  ortalama
tag   subset     feature_set                                                             
E2_P1 gorulen    A (ham)                      0.7667    0.9849  0.9763   0.9863    0.9286
                 A+AGG (referans)             0.8297    0.9879  0.9809   0.9904    0.9472
                 B (A+davranışsal)            0.3744    0.9834  0.9743   0.9849    0.8292
                 C (B+graf)                   0.9592    0.9953  0.9922   0.9956    0.9856
                 C (B+graf) [κ-mas

### Düzeltilmiş istatistiksel test (P1, B9 ile aynı yöntem)
Rolling-origin fold'ları değiştirilebilir (exchangeable) olmadığı için yalnızca P1 test edilir; P4 origin bazında tablo olarak raporlanır.

In [22]:
# ============================================================
# B9 — DUZELTILMIS ISTATISTIKSEL TEST
# ------------------------------------------------------------
# CV fold'lari bagimsiz degildir (egitim kumeleri ortusur); duz eslestirilmis
# t-testi varyansi olcmekte eksik kalir ve p-degerlerini oldugundan kucuk verir.
# Nadeau & Bengio (2003) duzeltilmis yeniden ornekleme t-testi:
#     t = mean(d) / sqrt((1/k + n_test/n_train) * var(d)),  df = k - 1
# 10 model ayni karsilastirmada test edildigi icin Holm (1979) duzeltmesi uygulanir.
# Protokol 3 tek bolme oldugu icin bu testle sinanmaz.
# ============================================================
B9_COMPARISONS = [('C (B+graf)', 'B (A+davranışsal)'), ('D (A+graf)', 'A (ham)'), ('B (A+davranışsal)', 'A (ham)')]
B9_METRICS     = ['f1', 'pr_auc']
B9_ALPHA       = 0.05

def corrected_resampled_ttest(a, b, test_train_ratio):
    d = np.asarray(a, float) - np.asarray(b, float); k = len(d); v = d.var(ddof=1)
    if v == 0:
        return d.mean(), np.nan, 1.0
    t = d.mean() / np.sqrt((1.0 / k + test_train_ratio) * v)
    return d.mean(), t, float(2 * stats.t.sf(abs(t), df=k - 1))

def holm_adjust(pvals):
    p = np.nan_to_num(np.asarray(pvals, float), nan=1.0); m = len(p)
    adj, run = np.empty(m), 0.0
    for r, i in enumerate(np.argsort(p)):
        run = max(run, min(1.0, (m - r) * p[i])); adj[i] = run
    return adj

def significance_table(res, splits, protocol):
    ratio = float(np.mean([len(te) / len(tr) for tr, te in splits]))
    rows = []
    for s1, s2 in B9_COMPARISONS:
        for met in B9_METRICS:
            block = []
            for m in sorted(res['model'].unique()):
                a = res[(res.model == m) & (res.feature_set == s1)].sort_values('fold')[met].values
                b = res[(res.model == m) & (res.feature_set == s2)].sort_values('fold')[met].values
                if len(a) < 2 or len(a) != len(b):
                    continue
                diff, t, p = corrected_resampled_ttest(a, b, ratio)
                p_naive = stats.ttest_rel(a, b).pvalue if np.std(a - b) > 0 else 1.0
                block.append({'protocol': protocol, 'comparison': f'{s1} vs {s2}', 'metric': met, 'model': m,
                              'mean_diff': diff, 't_corr': t, 'p_naive': p_naive, 'p_corr': p})
            for r, pa in zip(block, holm_adjust([r['p_corr'] for r in block])):
                r['p_holm'] = pa
                r['naive_sig'] = r['p_naive'] < B9_ALPHA
                r['holm_sig']  = pa < B9_ALPHA
            rows += block
    return pd.DataFrame(rows)


B9_COMPARISONS = [('C (B+graf)', 'B (A+davranışsal)'), ('D (A+graf)', 'A (ham)'), ('B (A+davranışsal)', 'A (ham)'),
                  ('Router C (B+graf) | A (ham)', 'A (ham)'), ('C (B+graf) [κ-mask]', 'C (B+graf)')]
if 'P1' in e2_res:
    sig_e2 = significance_table(e2_res['P1'], [(tr, ev) for tr, ev, _ in PROTOCOLS['P1'][1]], 'P1')
    sig_e2.to_csv(f'elliptic_significance_{PIPELINE_VERSION}.csv', index=False)
    print(sig_e2.groupby(['comparison', 'metric']).agg(ort_fark=('mean_diff', 'mean'), anlamli_naive=('naive_sig', 'sum'),
                                                        anlamli_holm=('holm_sig', 'sum'), model=('model', 'count')).round(4).to_string())


                                               ort_fark  anlamli_naive  anlamli_holm  model
comparison                             metric                                              
B (A+davranışsal) vs A (ham)           f1       -0.0655              1             1      4
                                       pr_auc   -0.0908              3             2      4
C (B+graf) [κ-mask] vs C (B+graf)      f1        0.0008              0             0      4
                                       pr_auc    0.0064              2             1      4
C (B+graf) vs B (A+davranışsal)        f1        0.1097              4             4      4
                                       pr_auc    0.1489              4             4      4
D (A+graf) vs A (ham)                  f1        0.0527              4             4      4
                                       pr_auc    0.0651              4             4      4
Router C (B+graf) | A (ham) vs A (ham) f1        0.0368              4          

## 13. Tablolar

In [23]:
tables_e = {'E_audit': audit_table if 'audit_table' in globals() else None,
            'E_f1': e2_tables['f1'], 'E_pr_auc': e2_tables['pr_auc'],
            'E_kappa_stratified': e2_strat if 'e2_strat' in globals() else None,
            'E_significance_P1': sig_e2 if 'sig_e2' in globals() else None,
            'E_wallet_feature_check': wallet_feature_check if 'wallet_feature_check' in globals() else None}
with pd.ExcelWriter('paper_tables_elliptic.xlsx') as _w:
    for _n, _t in tables_e.items():
        if _t is not None:
            _t.to_excel(_w, sheet_name=_n[:31])
print('paper_tables_elliptic.xlsx kaydedildi:', [k for k, v in tables_e.items() if v is not None])


paper_tables_elliptic.xlsx kaydedildi: ['E_audit', 'E_f1', 'E_pr_auc', 'E_kappa_stratified', 'E_significance_P1']


## 14. E3 — GNN karşılaştırması (GCN, GraphSAGE)

Kurulum (Colab'da torch hazır): `!pip -q install torch_geometric`

- **Graf:** tx–tx para akışı grafı (yönsüz). Kenarlar adım içinde olduğu için zamansal protokollerde eğitim grafı (adım ≤ kesim) ile test adımlarının grafları ayrıktır → model test adımlarına **inductive** uygulanır.
- **Eğitim:** yalnızca yapı bölümündeki düğümlerle (rastgele CV'de değerlendirme düğümleri eğitim grafından çıkarılır). Etiketli eğitim düğümlerinin %20'si doğrulama: erken durdurma (PR-AUC) ve eşik seçimi burada — tablo modellerdeki `fit_predict` ile aynı ilke.
- **Öznitelikler:** A (işlemin kendi özellikleri) ve A+AGG (Elliptic'in hazır komşuluk aggregate'leri; literatürdeki "AF").
- **Not:** GNN yalnızca tx–tx grafını kullanır; cüzdan–işlem grafı üzerinde heterojen GNN bu aşamanın kapsamı dışında.


In [24]:
!pip -q install torch_geometric
import torch, torch.nn.functional as F
from torch_geometric.nn import GCNConv, SAGEConv
from torch_geometric.utils import to_undirected, subgraph

RUN_E3        = False   # E3 sonuclari gunlukte; checkpoint varsa asagida yuklenir. Yeniden kosu: True
GNN_KINDS     = ['GCN', 'GraphSAGE']
GNN_FEATS     = {'A': FEATURES_A, 'A+AGG': FEATURES_A + AGG_FEATS}
GNN_PROTOCOLS = ['P3', 'P4', 'P1']
GNN_SEEDS     = [0, 1, 2]
GNN_HIDDEN, GNN_LAYERS, GNN_DROPOUT = 64, 2, 0.5
GNN_LR, GNN_WD, GNN_EPOCHS, GNN_PATIENCE = 0.01, 5e-4, 300, 50
GNN_CLIP = 10.0                                   # olceklenmis oznitelik kirpma (asiri degerler)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

EDGE_INDEX = to_undirected(torch.tensor(np.vstack([TX_EDGES.a.values, TX_EDGES.b.values]), dtype=torch.long),
                           num_nodes=N_TX).to(DEVICE)
print(f'GNN cihazi: {DEVICE} | dugum {N_TX:,} | yonsuz kenar {EDGE_INDEX.shape[1]:,}')

class GNN(torch.nn.Module):
    def __init__(self, kind, d_in, hidden=GNN_HIDDEN, layers=GNN_LAYERS, dropout=GNN_DROPOUT):
        super().__init__()
        make = {'GCN': lambda i, o: GCNConv(i, o), 'GraphSAGE': lambda i, o: SAGEConv(i, o)}[kind]
        dims = [d_in] + [hidden] * layers
        self.convs = torch.nn.ModuleList([make(dims[k], dims[k + 1]) for k in range(layers)])
        self.head, self.dropout = torch.nn.Linear(hidden, 1), dropout
    def forward(self, x, ei):
        for conv in self.convs:
            x = F.dropout(F.relu(conv(x, ei)), p=self.dropout, training=self.training)
        return self.head(x).squeeze(-1)

def gnn_fold(kind, cols, tr_c, ev_c, st_c, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    X = np.nan_to_num(TXF[cols].values.astype(np.float32))
    X = np.clip(StandardScaler().fit(X[tr_c]).transform(X), -GNN_CLIP, GNN_CLIP).astype(np.float32)
    x = torch.tensor(X, device=DEVICE)
    st = np.sort(np.asarray(st_c)); pos = np.full(N_TX, -1); pos[st] = np.arange(len(st))
    ei_tr, _ = subgraph(torch.tensor(st, device=DEVICE), EDGE_INDEX, relabel_nodes=True, num_nodes=N_TX)
    x_tr = x[torch.tensor(st, device=DEVICE)]
    y_all = torch.tensor(np.nan_to_num(Y), dtype=torch.float32, device=DEVICE)
    tr_pos = pos[np.asarray(tr_c)]
    from sklearn.model_selection import train_test_split
    a_pos, v_pos = train_test_split(tr_pos, test_size=VAL_FRAC, stratify=Y[st[tr_pos]], random_state=seed)
    a_t, v_t = torch.tensor(a_pos, device=DEVICE), torch.tensor(v_pos, device=DEVICE)
    y_st = y_all[torch.tensor(st, device=DEVICE)]
    pw = torch.tensor((y_st[a_t] == 0).sum().item() / max(1, (y_st[a_t] == 1).sum().item()), device=DEVICE)
    model = GNN(kind, X.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=GNN_LR, weight_decay=GNN_WD)
    best, best_state, wait, t0 = -1.0, None, 0, time.time()
    yv = y_st[v_t].cpu().numpy()
    for ep in range(GNN_EPOCHS):
        model.train(); opt.zero_grad()
        out = model(x_tr, ei_tr)
        loss = F.binary_cross_entropy_with_logits(out[a_t], y_st[a_t], pos_weight=pw)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            sv = torch.sigmoid(model(x_tr, ei_tr)[v_t]).cpu().numpy()
        score = average_precision_score(yv, sv) if len(np.unique(yv)) > 1 else 0.0
        if score > best:
            best, best_state, wait, best_sv, best_ep = score, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0, sv, ep
        else:
            wait += 1
            if wait >= GNN_PATIENCE: break
    train_s = time.time() - t0
    model.load_state_dict(best_state); model.eval()
    thr = _pick_threshold(yv, best_sv)
    t1 = time.time()
    with torch.no_grad():
        ys = torch.sigmoid(model(x, EDGE_INDEX)).cpu().numpy()[np.asarray(ev_c)]
    infer_s = time.time() - t1
    return ys, (ys >= thr).astype(int), thr, {'train_s': train_s, 'infer_s': infer_s, 'best_epoch': best_ep, 'val_pr_auc': best}
print('GNN yardimcilari hazir.')


GNN cihazi: cpu | dugum 203,769 | yonsuz kenar 468,710
GNN yardimcilari hazir.


In [25]:
def run_gnn(key):
    label, splits = PROTOCOLS[key]
    tag = f'E3_GNN_{key}'; rp = os.path.join(CHECKPOINT_DIR, f'{tag}_results.csv')
    rows, done = [], set()
    if RESUME and os.path.exists(rp):
        prev = pd.read_csv(rp); rows = prev.to_dict('records'); done = set(prev['fold'].astype(int))
        print(f'  [DEVAM] {tag}: tamamlanmis fold = {sorted(done)}', flush=True)
    for fold, (tr_c, ev_c, st_c) in enumerate(splits, start=1):
        if fold in done: continue
        t_fold = time.time()
        _, f_te, tinfo = compute_fold_features(tr_c, ev_c, st_c, verbose=False)     # yalnizca kappa bayraklari icin
        y_te = Y[np.asarray(ev_c)].astype(int); seen = f_te['_ent_seen'].values.astype(np.int8)
        names, S, P = [], [], []
        for kind in GNN_KINDS:
            for fname, cols in GNN_FEATS.items():
                for seed in GNN_SEEDS:
                    ys, yp, thr, info = gnn_fold(kind, cols, tr_c, ev_c, st_c, seed)
                    r = full_metrics(y_te, yp, ys)
                    r.update({'protocol': label, 'fold': fold, 'fold_label': FOLD_LABEL.get(key, {}).get(fold, str(fold)),
                              'feature_set': f'{kind} ({fname})', 'model': 'GNN', 'seed': seed, 'threshold': thr,
                              'kappa_ent': tinfo['kappa'], 'n_eval': len(y_te), 'n_pos_eval': int(y_te.sum()), **info})
                    rows.append(r); names.append(f'{kind} ({fname})||GNN-s{seed}'); S.append(ys); P.append(yp)
        np.savez_compressed(os.path.join(CHECKPOINT_DIR, f'{tag}_fold{fold}_preds.npz'), names=np.array(names),
                            scores=np.vstack(S).astype(np.float32), preds=np.vstack(P).astype(np.int8),
                            y=y_te.astype(np.int8), seen_ent=seen, step=STEP[np.asarray(ev_c)].astype(np.int16))
        pd.DataFrame(rows).to_csv(rp, index=False)
        print(f'  [{label}] GNN fold {fold}/{len(splits)} tamam ({time.time()-t_fold:.0f}s)', flush=True)
    return pd.DataFrame(rows)

if not RUN_E3:
    import glob
    _e3f = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'E3_GNN_P*_results.csv')))
    if _e3f:
        E3 = pd.concat([pd.read_csv(f) for f in _e3f], ignore_index=True)
        print(f'E3 GNN sonuclari checkpoint\'ten yuklendi: {len(E3)} satir ({len(_e3f)} dosya)')
    else:
        print('E3 GNN checkpoint\'i yok; GNN karsilastirma tablosu atlanacak (RUN_E3 = True ile yeniden kosulur).')
if RUN_E3:
    E3 = pd.concat([run_gnn(k) for k in GNN_PROTOCOLS], ignore_index=True)
    E3.to_csv(f'elliptic_e3_gnn_{PIPELINE_VERSION}.csv', index=False)
    _g = E3.groupby(['protocol', 'feature_set'])[['f1', 'pr_auc', 'auc_roc', 'train_s', 'infer_s', 'best_epoch']]
    print('=' * 100); print('E3 — GNN (tohum ve fold ortalamasi; parantez: tohumlar arasi std)'); print('=' * 100)
    _m = _g.mean().round(4)
    _s = E3.groupby(['protocol', 'feature_set', 'fold'])[['f1', 'pr_auc']].mean().groupby(level=[0, 1]).std().round(4)
    _sd = E3.groupby(['protocol', 'feature_set', 'seed'])[['f1', 'pr_auc']].mean().groupby(level=[0, 1]).std().round(4)
    _m['f1_seed_std'], _m['pr_auc_seed_std'] = _sd['f1'], _sd['pr_auc']
    print(_m.to_string())


E3 GNN sonuclari checkpoint'ten yuklendi: 132 satir (3 dosya)


### Tablo modeller × GNN karşılaştırması (aynı protokol, aynı fold'lar)

In [26]:
if 'E3' in globals():
    COMPARE_TABULAR_MODELS = ['LightGBM', 'XGBoost']
    COMPARE_TABULAR_SETS   = ['A (ham)', 'A+AGG (referans)', 'C (B+graf)', 'D (A+graf)', 'C (B+graf) [κ-mask]',
                              'Router C (B+graf) | A (ham)']
    import glob
    _tab = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'E2_P*_results.csv')))], ignore_index=True)
    _tab = _tab[_tab.model.isin(COMPARE_TABULAR_MODELS) & _tab.feature_set.isin(COMPARE_TABULAR_SETS)].copy()
    _tab['method'] = _tab['model'] + ' · ' + _tab['feature_set']
    _gnn = E3.copy(); _gnn['method'] = _gnn['feature_set']
    _all = pd.concat([_tab[['protocol', 'fold', 'method', 'f1', 'pr_auc']], _gnn[['protocol', 'fold', 'method', 'f1', 'pr_auc']]])
    _order = [PROTOCOLS[k][0] for k in GNN_PROTOCOLS]
    compare_e3 = _all.pivot_table(index='method', columns='protocol', values=['pr_auc', 'f1'], aggfunc='mean')
    compare_e3 = compare_e3.reindex(columns=pd.MultiIndex.from_product([['pr_auc', 'f1'], _order])).round(4)
    print('=' * 120); print('Tablo modeller vs GNN — PR-AUC ve F1 (fold ortalamasi; GNN tohum ortalamasi)'); print('=' * 120)
    print(compare_e3.sort_values(('pr_auc', _order[0]), ascending=False).to_string())

    # kappa-ayrilmis: GNN tahmin dosyalarindan (alt grup prevalansiyla)
    _rows = []
    for _f in sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'E3_GNN_P*_fold*_preds.npz'))):
        _z = np.load(_f); _tag = os.path.basename(_f).split('_fold')[0]
        _y, _se = _z['y'].astype(int), _z['seen_ent'].astype(bool)
        for _k, _name in enumerate(_z['names']):
            _set = str(_name).split('||')[0]
            for _sub, _mk in [('gorulen', _se), ('gorulmeyen', ~_se)]:
                if _mk.sum() and len(np.unique(_y[_mk])) > 1:
                    _rows.append({'tag': _tag, 'subset': _sub, 'feature_set': _set, 'n_pos': int(_y[_mk].sum()),
                                  'pr_auc': average_precision_score(_y[_mk], _z['scores'][_k][_mk])})
    e3_strat = pd.DataFrame(_rows)
    if len(e3_strat):
        print(); print('GNN — kappa-ayrilmis PR-AUC (tohum ve fold ortalamasi)')
        print(e3_strat.pivot_table(index=['tag', 'subset'], columns='feature_set', values='pr_auc').round(4).to_string())

else:
    print('E3 sonucu yok: tablo modeller × GNN karsilastirmasi atlandi.')


Tablo modeller vs GNN — PR-AUC ve F1 (fold ortalamasi; GNN tohum ortalamasi)
                                                      pr_auc                                                    f1                                
                                       Temporal 1-34 / 35-49 Rolling-origin Random 5-fold CV Temporal 1-34 / 35-49 Rolling-origin Random 5-fold CV
method                                                                                                                                            
LightGBM · A+AGG (referans)                           0.7979         0.8195           0.9835                0.7971         0.7390           0.9474
XGBoost · A+AGG (referans)                            0.7960         0.8511           0.9859                0.8065         0.7292           0.9513
LightGBM · Router C (B+graf) | A (ham)                0.7908         0.7899           0.9885                0.7578         0.7404           0.9573
LightGBM · A (ham)                       

### Gecikme (latency): öznitelik + tahmin süresi vs GNN çıkarımı
Aynı P3 bölmesinde, 1.000 işlem başına süre. GNN için tam graf ileri geçişi ve yalnızca test adımlarının alt grafı ayrı ölçülür.

In [27]:
RUN_LATENCY = False   # E3 sonucu gunlukte; yeniden olcum icin True
if RUN_LATENCY:
    _tr, _ev, _S = split_temporal(TEMPORAL_CUT)
    t0 = time.time(); _ftr, _fte, _ti = compute_fold_features(_tr, _ev, _S, verbose=False); feat_s = time.time() - t0
    _lat = []
    for _s in ['A (ham)', 'C (B+graf)']:
        _cols = E2_SETS[_s]
        _m = get_models()['LightGBM'].fit(_X(_ftr, _cols), _ftr['y'].values.astype(int))
        _Xte = _X(_fte, _cols); t0 = time.time(); _m.predict_proba(_Xte); pred_s = time.time() - t0
        _lat.append({'yontem': f'LightGBM · {_s}', 'oznitelik_s': feat_s if _s != 'A (ham)' else 0.0, 'tahmin_s': pred_s,
                     'ms_per_1k': ((feat_s if _s != 'A (ham)' else 0.0) + pred_s) * 1e6 / len(_ev)})
    for _k in GNN_KINDS:
        _cols = GNN_FEATS['A']
        _X0 = np.clip(StandardScaler().fit(np.nan_to_num(TXF[_cols].values[_tr])).transform(np.nan_to_num(TXF[_cols].values)), -GNN_CLIP, GNN_CLIP)
        _x = torch.tensor(_X0.astype(np.float32), device=DEVICE); _mdl = GNN(_k, len(_cols)).to(DEVICE).eval()
        with torch.no_grad():
            t0 = time.time(); _mdl(_x, EDGE_INDEX); full_s = time.time() - t0
            _evs = np.where(STEP > TEMPORAL_CUT)[0]
            _ei, _ = subgraph(torch.tensor(_evs, device=DEVICE), EDGE_INDEX, relabel_nodes=True, num_nodes=N_TX)
            t0 = time.time(); _mdl(_x[torch.tensor(_evs, device=DEVICE)], _ei); sub_s = time.time() - t0
        _lat.append({'yontem': f'{_k} · A (tam graf)', 'oznitelik_s': 0.0, 'tahmin_s': full_s, 'ms_per_1k': full_s * 1e6 / len(_ev)})
        _lat.append({'yontem': f'{_k} · A (test adimlari alt grafi)', 'oznitelik_s': 0.0, 'tahmin_s': sub_s, 'ms_per_1k': sub_s * 1e6 / len(_ev)})
    latency_e3 = pd.DataFrame(_lat)
    print(f'Cihaz: {DEVICE} | test islemi: {len(_ev):,} | oznitelik hesaplama (cuzdan, tum fold): {feat_s:.1f}s')
    print(latency_e3.round(4).to_string(index=False))
    print('Not: cuzdan oznitelikleri canli sistemde artimli guncellenebilir; buradaki sure tum fold icin toplu hesaplamadir.')


In [28]:
for _n, _v in [('E3_gnn', 'E3'), ('E3_compare', 'compare_e3'), ('E3_gnn_kappa', 'e3_strat'), ('E3_latency', 'latency_e3')]:
    if _v in globals():
        tables_e[_n] = globals()[_v]
with pd.ExcelWriter('paper_tables_elliptic.xlsx') as _w:
    for _n, _t in tables_e.items():
        if _t is not None:
            _t.to_excel(_w, sheet_name=_n[:31])
print('paper_tables_elliptic.xlsx guncellendi:', [k for k, v in tables_e.items() if v is not None])


paper_tables_elliptic.xlsx guncellendi: ['E_audit', 'E_f1', 'E_pr_auc', 'E_kappa_stratified', 'E_significance_P1', 'E3_gnn', 'E3_compare', 'E3_gnn_kappa']


## 15. E4 — Güçlü baseline (A+AGG) üzerinde graf katkısı ve router

E2'de zamansal protokollerde en güçlü set Elliptic'in hazır komşuluk aggregate'lerini içeren **A+AGG** idi; önerilen router ise
dar Set A üzerine kuruluydu. Burada cüzdan (davranışsal + graf) ve tx-grafı öznitelikleri A+AGG'nin **üstüne** eklenir ve
router güçlü baseline'a geri döner: giriş cüzdanı eğitim grafında görüldüyse A+AGG+graf modeli, görülmediyse A+AGG modeli.

In [29]:
import glob
RUN_E4       = {'P1': True, 'P3': True, 'P4': True}
E4_MODELS    = E2_MODELS
E4_BASE      = 'A+AGG (referans)'
_AA          = SET_REF[E4_BASE]
E4_SETS      = {E4_BASE: _AA,
                'A+AGG+B': _AA + BEHAVIORAL,
                'A+AGG+C': _AA + BEHAVIORAL + GRAPH_FEATS,
                'A+AGG+D': _AA + GRAPH_FEATS}
E4_KAPPA_MASK = ['A+AGG+C', 'A+AGG+D']
E4_ROUTERS    = [('A+AGG+C', E4_BASE), ('A+AGG+D', E4_BASE)]
KAPPA_MIN_POS = 30          # kappa-ayrilmis hucrelerde fold basina asgari pozitif sayisi
for _k, _v in E4_SETS.items():
    print(f'  {_k:18s}: {len(_v):3d} oznitelik')

e4_res = {}
for _k, _on in RUN_E4.items():
    if _on:
        print(f'=== E4 {_k}: {PROTOCOLS[_k][0]}', flush=True)
        e4_res[_k], _ = run_protocol_e(_k, model_names=E4_MODELS, feature_sets=E4_SETS,
                                       kappa_mask_sets=E4_KAPPA_MASK, router_pairs=E4_ROUTERS, tag=f'E4_{_k}')
E4 = pd.concat(e4_res.values(), ignore_index=True)
E4.to_csv(f'elliptic_e4_results_{PIPELINE_VERSION}.csv', index=False)
print(f'E4: {len(E4)} satir')


  A+AGG (referans)  : 180 oznitelik
  A+AGG+B           : 200 oznitelik
  A+AGG+C           : 212 oznitelik
  A+AGG+D           : 192 oznitelik
=== E4 P1: Random 5-fold CV
  [DEVAM] E4_P1: tamamlanmis fold = [1, 2, 3, 4, 5]
=== E4 P3: Temporal 1-34 / 35-49
  [DEVAM] E4_P3: tamamlanmis fold = [1]
=== E4 P4: Rolling-origin
  [DEVAM] E4_P4: tamamlanmis fold = [1, 2, 3, 4, 5]
E4: 352 satir


In [30]:
SHORT4 = {E4_BASE: 'AA', 'A+AGG+B': 'AA+B', 'A+AGG+C': 'AA+C', 'A+AGG+D': 'AA+D',
          'A+AGG+C [κ-mask]': 'AA+C κ', 'A+AGG+D [κ-mask]': 'AA+D κ',
          f'Router A+AGG+C | {E4_BASE}': 'Router AA+C|AA', f'Router A+AGG+D | {E4_BASE}': 'Router AA+D|AA'}
E4_DIFFS = {'B−AA': ('AA+B', 'AA'), 'D−AA': ('AA+D', 'AA'), 'C−AA': ('AA+C', 'AA'), 'C−B': ('AA+C', 'AA+B'),
            'Cκ−C': ('AA+C κ', 'AA+C'), 'RouterC−AA': ('Router AA+C|AA', 'AA'), 'RouterD−AA': ('Router AA+D|AA', 'AA')}

def e4_table(res, met):
    p = res.pivot_table(index=['protocol', 'model'], columns='feature_set', values=met)
    o = pd.DataFrame(index=p.index)
    o['kappa_ent'] = res.groupby(['protocol', 'model'])['kappa_ent'].mean()
    for full, short in SHORT4.items():
        if full in p: o[short] = p[full]
    for name, (a, b) in E4_DIFFS.items():
        if a in o and b in o: o[name] = o[a] - o[b]
    mean = o.groupby(level='protocol').mean(); mean.index = pd.MultiIndex.from_tuples([(i, 'ORTALAMA') for i in mean.index])
    order = list(dict.fromkeys(res['protocol']))
    o = pd.concat([o, mean]); return o.loc[[ix for pr in order for ix in o.index if ix[0] == pr]].round(4)

e4_tables = {m: e4_table(E4, m) for m in ['pr_auc', 'f1']}
for _m, _t in e4_tables.items():
    print('=' * 140); print(f'E4 — {_m} (AA = A+AGG)'); print('=' * 140); print(_t.to_string()); print()

# E2 ile yan yana: dar baseline uzerindeki router vs guclu baseline uzerindeki router (ayni fold'lar)
_e2f = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'E2_P*_results.csv')))
if _e2f:
    _e2 = pd.concat([pd.read_csv(f) for f in _e2f], ignore_index=True)
    _e2 = _e2[_e2.feature_set.isin(['A (ham)', 'Router C (B+graf) | A (ham)', 'C (B+graf) [κ-mask]'])]
    _both = pd.concat([_e2[['protocol', 'model', 'fold', 'feature_set', 'pr_auc', 'f1']],
                       E4[['protocol', 'model', 'fold', 'feature_set', 'pr_auc', 'f1']]])
    _both['feature_set'] = _both['feature_set'].replace({**SHORT, **SHORT4})
    e4_vs_e2 = _both.pivot_table(index='feature_set', columns='protocol', values=['pr_auc', 'f1'], aggfunc='mean').round(4)
    print('=' * 120); print('Dar (A) vs guclu (AA) baseline — modellerin ortalamasi'); print('=' * 120)
    _sk = ('pr_auc', 'Rolling-origin') if ('pr_auc', 'Rolling-origin') in e4_vs_e2.columns else e4_vs_e2.columns[0]
    print(e4_vs_e2.sort_values(_sk, ascending=False).to_string())


E4 — pr_auc (AA = A+AGG)
                                kappa_ent      AA    AA+B    AA+C    AA+D  AA+C κ  AA+D κ  Router AA+C|AA  Router AA+D|AA    B−AA    D−AA    C−AA     C−B    Cκ−C  RouterC−AA  RouterD−AA
Random 5-fold CV      LR           0.8094  0.7991  0.4322  0.9430  0.9643  0.9612  0.9613          0.9243          0.9435 -0.3669  0.1651  0.1439  0.5108  0.0182      0.1252      0.1444
                      LightGBM     0.8094  0.9835  0.9831  0.9929  0.9932  0.9935  0.9935          0.9912          0.9915 -0.0004  0.0097  0.0094  0.0098  0.0006      0.0077      0.0080
                      RF           0.8094  0.9732  0.9701  0.9860  0.9867  0.9862  0.9864          0.9846          0.9847 -0.0031  0.0134  0.0127  0.0158  0.0003      0.0113      0.0114
                      XGBoost      0.8094  0.9859  0.9850  0.9937  0.9936  0.9940  0.9941          0.9924          0.9921 -0.0009  0.0077  0.0078  0.0087  0.0003      0.0065      0.0062
                      ORTALAMA     0.8094  0.

### E4 — Rolling-origin: origin bazında farklar

In [31]:
if 'P4' in e4_res:
    _r = e4_res['P4'].copy(); _r['feature_set'] = _r['feature_set'].replace(SHORT4)
    e4_p4 = {}
    for _m in ['pr_auc', 'f1']:
        _p = _r.pivot_table(index='fold_label', columns='feature_set', values=_m, aggfunc='mean')
        _o = pd.DataFrame({'kappa_ent': _r.groupby('fold_label')['kappa_ent'].first(),
                           'n_pos': _r.groupby('fold_label')['n_pos_eval'].first()})
        for _c in ['AA', 'AA+C', 'AA+D', 'AA+C κ', 'Router AA+C|AA', 'Router AA+D|AA']:
            if _c in _p: _o[_c] = _p[_c]
        for name, (a, b) in E4_DIFFS.items():
            if a in _p and b in _p: _o[name] = _p[a] - _p[b]
        _o = _o.reindex([f'origin {o}' for o in ROLLING_ORIGINS])
        e4_p4[_m] = _o.round(4)
        print('=' * 120); print(f'E4 rolling-origin — {_m} (modellerin ortalamasi)'); print('=' * 120); print(_o.round(4).to_string())
        # model bazinda: ortalama fark ve kazanan origin sayisi
        _pm = _r.pivot_table(index=['model', 'fold_label'], columns='feature_set', values=_m)
        _w = []
        for name, (a, b) in E4_DIFFS.items():
            if a in _pm and b in _pm:
                for _mdl in _pm.index.get_level_values('model').unique():
                    _d = (_pm.loc[_mdl, a] - _pm.loc[_mdl, b]).dropna()
                    _w.append({'karsilastirma': name, 'model': _mdl, 'ort_fark': _d.mean(), 'std': _d.std(ddof=1),
                               'kazanan_origin': f'{int((_d > 0).sum())}/{len(_d)}'})
        e4_p4[f'wins_{_m}'] = pd.DataFrame(_w).set_index(['karsilastirma', 'model']).round(4)
        print(); print(f'Model bazinda ({_m}): ortalama ± std, kazanan origin'); print(e4_p4[f'wins_{_m}'].to_string()); print()


E4 rolling-origin — pr_auc (modellerin ortalamasi)
            kappa_ent  n_pos      AA    AA+C    AA+D  AA+C κ  Router AA+C|AA  Router AA+D|AA    B−AA    D−AA    C−AA     C−B    Cκ−C  RouterC−AA  RouterD−AA
fold_label                                                                                                                                                  
origin 24      0.4148    652  0.9263  0.9631  0.9660  0.9725          0.9723          0.9723 -0.0352  0.0398  0.0368  0.0720  0.0094      0.0460      0.0460
origin 29      0.3518    591  0.8888  0.9266  0.9320  0.9496          0.9575          0.9566  0.0207  0.0433  0.0379  0.0171  0.0229      0.0688      0.0678
origin 34      0.3963    447  0.8418  0.8619  0.8705  0.8851          0.8885          0.8886 -0.0569  0.0287  0.0201  0.0770  0.0232      0.0467      0.0468
origin 39      0.2482    515  0.7626  0.7265  0.7444  0.7634          0.7709          0.7714 -0.0055 -0.0182 -0.0361 -0.0306  0.0369      0.0083      0.0088
origin 

### E4 — κ'ya göre ayrılmış metrikler (alt küme boyutu, pozitif sayısı, güvenilirlik)

In [32]:
_rows, _cnt = [], []
for _f in sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'E4_P*_fold*_preds.npz'))):
    _z = np.load(_f); _tag = os.path.basename(_f).split('_fold')[0]
    _fold = int(os.path.basename(_f).split('_fold')[1].split('_')[0])
    _y, _se = _z['y'].astype(int), _z['seen_ent'].astype(bool)
    for _sub, _mk in [('gorulen', _se), ('gorulmeyen', ~_se)]:
        _cnt.append({'tag': _tag, 'fold': _fold, 'subset': _sub, 'n': int(_mk.sum()), 'pozitif': int(_y[_mk].sum())})
    for _k, _name in enumerate(_z['names']):
        _set, _model = str(_name).split('||')
        for _sub, _mk in [('gorulen', _se), ('gorulmeyen', ~_se)]:
            if _mk.sum() == 0 or len(np.unique(_y[_mk])) < 2:
                continue
            _rows.append({'tag': _tag, 'fold': _fold, 'subset': _sub, 'feature_set': SHORT4.get(_set, _set), 'model': _model,
                          'n': int(_mk.sum()), 'pozitif': int(_y[_mk].sum()),
                          'pr_auc': average_precision_score(_y[_mk], _z['scores'][_k][_mk]),
                          'f1': f1_score(_y[_mk], _z['preds'][_k][_mk], zero_division=0)})
e4_strat = pd.DataFrame(_rows)
e4_counts = pd.DataFrame(_cnt)
if len(e4_counts):
    e4_counts = e4_counts.groupby(['tag', 'subset']).agg(fold=('fold', 'nunique'), n=('n', 'sum'), pozitif=('pozitif', 'sum'),
                                                         min_pozitif_fold=('pozitif', 'min'))
    e4_counts['prevalans_%'] = (100 * e4_counts['pozitif'] / e4_counts['n'].clip(lower=1)).round(2)
    e4_counts['guvenilir'] = np.where(e4_counts['min_pozitif_fold'] >= KAPPA_MIN_POS, 'evet', 'HAYIR')
    print(f'Alt kume boyutlari (KAPPA_MIN_POS={KAPPA_MIN_POS}):'); print(e4_counts.to_string()); print()
if len(e4_strat):
    e4_strat.to_csv(f'elliptic_e4_kappa_stratified_{PIPELINE_VERSION}.csv', index=False)
    for _m in ['pr_auc', 'f1']:
        _p = e4_strat.pivot_table(index=['tag', 'subset', 'feature_set'], columns='model', values=_m)
        _p['ortalama'] = _p.mean(axis=1)
        _g = e4_counts['guvenilir'].to_dict()
        _p['guvenilir'] = [_g.get((t, s), '?') for t, s, _ in _p.index]
        print('=' * 110); print(f'E4 — kappa-ayrilmis {_m}'); print('=' * 110); print(_p.round(4).to_string()); print()


Alt kume boyutlari (KAPPA_MIN_POS=30):
                  fold      n  pozitif  min_pozitif_fold  prevalans_% guvenilir
tag   subset                                                                   
E4_P1 gorulen        5  37690     3349               651         8.89      evet
      gorulmeyen     5   8874     1196               216        13.48      evet
E4_P3 gorulen        1   5031        5                 5         0.10     HAYIR
      gorulmeyen     1  11639     1078              1078         9.26      evet
E4_P4 gorulen        5   8199      361                 2         4.40     HAYIR
      gorulmeyen     5  14759     1965               114        13.31      evet

E4 — kappa-ayrilmis pr_auc
model                                LR  LightGBM      RF  XGBoost  ortalama guvenilir
tag   subset     feature_set                                                          
E4_P1 gorulen    AA              0.8304    0.9879  0.9809   0.9904    0.9474      evet
                 AA+B           

### E4 — Düzeltilmiş istatistiksel test (P1)

In [33]:
_b9_keep = B9_COMPARISONS
B9_COMPARISONS = [('A+AGG+C', 'A+AGG+B'), ('A+AGG+D', E4_BASE), ('A+AGG+B', E4_BASE),
                  (f'Router A+AGG+C | {E4_BASE}', E4_BASE), ('A+AGG+C [κ-mask]', 'A+AGG+C')]
if 'P1' in e4_res:
    sig_e4 = significance_table(e4_res['P1'], [(tr, ev) for tr, ev, _ in PROTOCOLS['P1'][1]], 'P1')
    sig_e4.to_csv(f'elliptic_e4_significance_{PIPELINE_VERSION}.csv', index=False)
    print(sig_e4.groupby(['comparison', 'metric']).agg(ort_fark=('mean_diff', 'mean'), anlamli_naive=('naive_sig', 'sum'),
                                                        anlamli_holm=('holm_sig', 'sum'), model=('model', 'count')).round(4).to_string())
B9_COMPARISONS = _b9_keep


                                                              ort_fark  anlamli_naive  anlamli_holm  model
comparison                                            metric                                              
A+AGG+B vs A+AGG (referans)                           f1       -0.0532              2             1      4
                                                      pr_auc   -0.0928              3             1      4
A+AGG+C [κ-mask] vs A+AGG+C                           f1        0.0054              1             0      4
                                                      pr_auc    0.0048              1             1      4
A+AGG+C vs A+AGG+B                                    f1        0.0947              4             4      4
                                                      pr_auc    0.1363              4             4      4
A+AGG+D vs A+AGG (referans)                           f1        0.0527              4             4      4
                                     

In [34]:
for _n, _v in [('E4_pr_auc', e4_tables['pr_auc']), ('E4_f1', e4_tables['f1']),
               ('E4_vs_E2', globals().get('e4_vs_e2')), ('E4_kappa_counts', globals().get('e4_counts')),
               ('E4_significance_P1', globals().get('sig_e4'))] + \
              [(f'E4_P4_{k}', v) for k, v in (globals().get('e4_p4') or {}).items()]:
    if _v is not None and len(_v):
        tables_e[_n] = _v
with pd.ExcelWriter('paper_tables_elliptic.xlsx') as _w:
    for _n, _t in tables_e.items():
        if _t is not None:
            _t.to_excel(_w, sheet_name=_n[:31])
print('paper_tables_elliptic.xlsx guncellendi:', [k for k, v in tables_e.items() if v is not None])


paper_tables_elliptic.xlsx guncellendi: ['E_audit', 'E_f1', 'E_pr_auc', 'E_kappa_stratified', 'E_significance_P1', 'E3_gnn', 'E3_compare', 'E3_gnn_kappa', 'E4_pr_auc', 'E4_f1', 'E4_vs_E2', 'E4_kappa_counts', 'E4_significance_P1', 'E4_P4_pr_auc', 'E4_P4_wins_pr_auc', 'E4_P4_f1', 'E4_P4_wins_f1']


## 16. E5 — Kalibre ve geçmiş eşikli router (P4)
Router iki modelin skorlarını birleştirir; ölçekleri farklı olabileceği için her model, eşiğin seçildiği iç doğrulama bölümünde **Platt ölçeklemesiyle**
kalibre edilir (model içi sıralamayı korur; isotonic karşılaştırma için raporlanır). İkili görüldü/görülmedi yerine **en az k geçmiş etkileşimi olan**
giriş cüzdanlarına sahip işlemler graf-koşullu modele yönlendirilir. F1, her modelin kendi eşiğiyle hesaplandığı için kalibrasyondan etkilenmez.

In [35]:
# ============================================================
# E5 — KALIBRE ve GECMIS ESIKLI ROUTER (P4 rolling-origin)
# ============================================================
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split

RUN_E5    = True
E5_MODELS = E2_MODELS
E5_SETS   = {'A (ham)': FEATURE_SETS['A (ham)'], 'C (B+graf)': FEATURE_SETS['C (B+graf)'],
             E4_BASE: E4_SETS[E4_BASE], 'A+AGG+C': E4_SETS['A+AGG+C']}
E5_PAIRS  = [('C (B+graf)', 'A (ham)'), ('A+AGG+C', E4_BASE)]
E5_K      = [1, 2, 3, 5, 10]
E5_TAG    = 'E5_P4'
HIST_COLS = ['ent_nsend_max', 'ent_nrecv_max']   # giris cuzdanlarinin yapi bolumundeki etkilesim sayisi (en aktif cuzdan)
PLATT_SLOPES = []

def _proba(m, X):
    try:    return m.predict_proba(X)[:, 1]
    except Exception: return m.decision_function(X)

def _logit(s):
    s = np.clip(np.asarray(s, float), 1e-12, 1 - 1e-12); return np.log(s / (1 - s))

def fit_predict_cal(model, Xtr, ytr, Xte, model_name=None):
    """fit_predict ile ayni egitim, esik ve skorlar + ic dogrulamada fit edilen Platt ve isotonic kalibratorler."""
    assert THRESHOLD_MODE == 'tuned'
    sc = StandardScaler(); Xtr_s = sc.fit_transform(Xtr); Xte_s = sc.transform(Xte)
    Xa, Xv, ya, yv = train_test_split(Xtr_s, ytr, test_size=VAL_FRAC, stratify=ytr, random_state=SEED)
    model.fit(Xa, ya); sv = _proba(model, Xv)
    thr = _pick_threshold(yv, sv)
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(_logit(sv).reshape(-1, 1), yv)
    a, b = float(platt.coef_[0, 0]), float(platt.intercept_[0])
    iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip').fit(sv, yv)
    model.fit(Xtr_s, ytr); ys = _proba(model, Xte_s)
    cal = 1.0 / (1.0 + np.exp(-(a * _logit(ys) + b))) if a > 0 else ys
    PLATT_SLOPES.append(a)
    return (ys >= thr).astype(int), ys, cal, iso.predict(ys), thr

if RUN_E5:
    _rp = os.path.join(CHECKPOINT_DIR, f'{E5_TAG}_results.csv')
    e5_rows, _done = [], set()
    if RESUME and os.path.exists(_rp):
        _prev = pd.read_csv(_rp); e5_rows = _prev.to_dict('records'); _done = set(_prev['fold'].astype(int))
        print(f'  [DEVAM] {E5_TAG}: tamamlanmis fold = {sorted(_done)}')
    for fold, (tr_c, ev_c, st_c) in enumerate(PROTOCOLS['P4'][1], start=1):
        if fold in _done: continue
        t0 = time.time()
        f_tr, f_te, tinfo = compute_fold_features(tr_c, ev_c, st_c, verbose=False)
        y_tr, y_te = f_tr['y'].values.astype(int), f_te['y'].values.astype(int)
        seen = f_te[ROUTER_SEEN_COL].values.astype(bool)
        hist = np.where(seen, np.nan_to_num(f_te[HIST_COLS].astype(float).values).sum(axis=1), 0.0)
        out = {}
        for s, cols in E5_SETS.items():
            Xtr, Xte = _X(f_tr, cols), _X(f_te, cols)
            for m in E5_MODELS:
                out[(s, m)] = fit_predict_cal(get_models()[m], Xtr, y_tr, Xte, model_name=m)
        base = {'fold': fold, 'fold_label': FOLD_LABEL['P4'][fold], 'kappa_hat': float(seen.mean()),
                'n_eval': len(y_te), 'n_pos': int(y_te.sum())}
        for warm, cold in E5_PAIRS:
            for m in E5_MODELS:
                (pw, sw, cw, iw, _), (pc, sc_, cc, ic, _) = out[(warm, m)], out[(cold, m)]
                variants = {'base': (pc, sc_, cc), 'warm': (pw, sw, cw),
                            'router raw': (np.where(seen, pw, pc), np.where(seen, sw, sc_), None),
                            'router isotonic': (np.where(seen, pw, pc), np.where(seen, iw, ic), None)}
                for k in E5_K:
                    msk = hist >= k
                    variants[f'router k>={k}'] = (np.where(msk, pw, pc), np.where(msk, cw, cc), float(msk.mean()))
                for vname, (yp, ys, extra) in variants.items():
                    r = full_metrics(y_te, yp.astype(int), ys)
                    r.update({**base, 'pair': f'{warm} | {cold}', 'model': m, 'variant': vname,
                              'routed_share': extra if isinstance(extra, float) else np.nan})
                    if vname in ('base', 'warm'):
                        r['pr_auc_platt'] = average_precision_score(y_te, extra)
                    e5_rows.append(r)
        pd.DataFrame(e5_rows).to_csv(_rp, index=False)
        print(f'  [E5] {FOLD_LABEL["P4"][fold]} tamam ({time.time() - t0:.0f}s)', flush=True)
        del f_tr, f_te; gc.collect()
    E5 = pd.DataFrame(e5_rows)
    E5.to_csv(f'elliptic_e5_router_{PIPELINE_VERSION}.csv', index=False)
    print(f'E5: {len(E5)} satir')


  [DEVAM] E5_P4: tamamlanmis fold = [1, 2, 3, 4, 5]
E5: 360 satir


In [36]:
# ============================================================
# E5 — TABLOLAR ve KONTROL
# ============================================================
if RUN_E5:
    _ref = pd.concat([d for k, d in [('e2', globals().get('e2_res', {}).get('P4')), ('e4', globals().get('e4_res', {}).get('P4'))] if d is not None])
    for warm, cold in E5_PAIRS:
        for m in E5_MODELS:
            a = _ref[(_ref.feature_set == f'Router {warm} | {cold}') & (_ref.model == m)].sort_values('fold')['pr_auc'].values
            b = E5[(E5.pair == f'{warm} | {cold}') & (E5.model == m) & (E5.variant == 'router raw')].sort_values('fold')['pr_auc'].values
            if len(a) == len(b) and len(a):
                print(f'  kontrol {m:8s} {warm[:10]:10s}: ham router == E2/E4  max|fark| = {np.abs(a - b).max():.2e}')
    _e5_order = ['base', 'warm', 'router raw', 'router isotonic'] + [f'router k>={k}' for k in E5_K]
    e5_tables = {}
    for met in ['pr_auc', 'f1']:
        p = E5.pivot_table(index=['pair', 'model'], columns='variant', values=met).reindex(columns=_e5_order)
        e5_tables[f'mean_{met}'] = p.round(4)
        print('=' * 130); print(f'E5 — {met} (origin ortalamasi)'); print('=' * 130); print(p.round(4).to_string()); print()
        rows = []
        for (pair, m), g in E5.groupby(['pair', 'model']):
            pv = g.pivot_table(index='fold', columns='variant', values=met)
            for v in _e5_order[1:]:
                dd = (pv[v] - pv['base']).dropna()
                rows.append({'pair': pair, 'model': m, 'variant': v, 'ort_fark': dd.mean(), 'std': dd.std(ddof=1),
                             'kazanan_origin': f'{int((dd > 0).sum())}/{len(dd)}'})
        e5_tables[f'diff_{met}'] = pd.DataFrame(rows).set_index(['pair', 'model', 'variant']).round(4)
        print(f'E5 — {met}: taban modele gore fark (ortalama ± std, kazanan origin)'); print(e5_tables[f'diff_{met}'].to_string()); print()
    sh = E5[E5.variant.str.startswith('router k')].pivot_table(index='variant', columns='fold_label', values='routed_share', aggfunc='mean')
    e5_tables['routed_share'] = sh.round(3)
    print('Graf-kosullu modele yonlendirilen pay:'); print(sh.round(3).to_string())
    print(f'Platt egimleri: min {min(PLATT_SLOPES):.3f}, medyan {np.median(PLATT_SLOPES):.3f}, pozitif olmayan {sum(a <= 0 for a in PLATT_SLOPES)}/{len(PLATT_SLOPES)}' if PLATT_SLOPES else '')
    for _k, _t in e5_tables.items():
        tables_e[f'E5_{_k}'] = _t
    with pd.ExcelWriter('paper_tables_elliptic.xlsx') as _w:
        for _n, _t in tables_e.items():
            if _t is not None:
                _t.to_excel(_w, sheet_name=_n[:31])
    print('paper_tables_elliptic.xlsx guncellendi.')


  kontrol LR       C (B+graf): ham router == E2/E4  max|fark| = 5.27e-03
  kontrol RF       C (B+graf): ham router == E2/E4  max|fark| = 0.00e+00
  kontrol XGBoost  C (B+graf): ham router == E2/E4  max|fark| = 0.00e+00
  kontrol LightGBM C (B+graf): ham router == E2/E4  max|fark| = 0.00e+00
  kontrol LR       A+AGG+C   : ham router == E2/E4  max|fark| = 0.00e+00
  kontrol RF       A+AGG+C   : ham router == E2/E4  max|fark| = 2.22e-16
  kontrol XGBoost  A+AGG+C   : ham router == E2/E4  max|fark| = 0.00e+00
  kontrol LightGBM A+AGG+C   : ham router == E2/E4  max|fark| = 0.00e+00
E5 — pr_auc (origin ortalamasi)
variant                                base    warm  router raw  router isotonic  router k>=1  router k>=2  router k>=3  router k>=5  router k>=10
pair                       model                                                                                                                  
A+AGG+C | A+AGG (referans) LR        0.5120  0.5719      0.6315           0.6181       0.6

## 17. E6 — Kapasite, karşı taraf router'ı, prefix öznitelikler ve belirsizlik (P4)
**Kapasite:** 50 ağaç yerine erken durdurmalı topluluk (ağaç sayısı iç doğrulamada PR-AUC ile). **Karşı taraf:** giriş cüzdanı görülmeyen
işlemde çıkış cüzdanlarının geçmişi ve işlem grafı yine de bilinebilir. **Prefix:** standart kurulumda eğitim işlemlerinin cüzdan öznitelikleri
origin anındaki yapıdan hesaplanır; burada her eğitim işlemi yalnızca kendi adımından önceki yapı ve etiketlerle betimlenir
(yapı `E6_PREFIX_REFRESH` adımda bir yenilenir; ilk adımın işlemleri geçmiş olmadığından bu varyantta eğitimden çıkarılır ve taban model
aynı satırlarla yeniden eğitilir). Değerlendirme satırları değişmez.

In [37]:
# ============================================================
# E6 — KAPASITE, KARSI TARAF ROUTER'I, PREFIX ve BOOTSTRAP (P4)
# ============================================================
import lightgbm as lgb
from sklearn.base import clone
from sklearn.model_selection import train_test_split

RUN_E6            = True
E6_TAG            = 'E6_P4'
E6_MODELS         = E2_MODELS                  # ref, karsi taraf ve prefix varyantlari
E6_ES_MODELS      = ['XGBoost', 'LightGBM']    # kapasite varyanti
E6_PAIRS          = E5_PAIRS                   # (graf-kosullu, taban)
E6_SETS           = E5_SETS
E6_ES_MAX         = 2000
E6_ES_LR          = 0.1
E6_ES_PAT         = 50
E6_PREFIX_REFRESH = 5                          # prefix yapisi kac adimda bir yenilenir
E6_PREFIX_START   = 2                          # bu adimdan onceki egitim islemleri prefix varyantinda kullanilmaz
E6_BOOT           = 200
CP_COLS_E = TX_GRAPH + side_entity_cols('cp')  # varlik gorulmese de bilinebilen oznitelikler
print(f'Karsi taraf + islem grafi oznitelikleri ({len(CP_COLS_E)}): {CP_COLS_E}')

def fit_predict_es(model_name, Xtr, ytr, Xte):
    """fit_predict ile ayni protokol; agac sayisi ic dogrulamada PR-AUC ile erken durdurmayla secilir."""
    sc = StandardScaler(); Xtr_s = sc.fit_transform(Xtr); Xte_s = sc.transform(Xte)
    Xa, Xv, ya, yv = train_test_split(Xtr_s, ytr, test_size=VAL_FRAC, stratify=ytr, random_state=SEED)
    base = get_models()[model_name]
    if model_name == 'LightGBM':
        m = clone(base).set_params(n_estimators=E6_ES_MAX, learning_rate=E6_ES_LR)
        m.fit(Xa, ya, eval_set=[(Xv, yv)], eval_metric='average_precision',
              callbacks=[lgb.early_stopping(E6_ES_PAT, first_metric_only=True, verbose=False)])
        n_best = int(m.best_iteration_ or E6_ES_MAX); sv = m.predict_proba(Xv, num_iteration=n_best)[:, 1]
    elif model_name == 'XGBoost':
        m = clone(base).set_params(n_estimators=E6_ES_MAX, learning_rate=E6_ES_LR, eval_metric='aucpr',
                                   early_stopping_rounds=E6_ES_PAT)
        m.fit(Xa, ya, eval_set=[(Xv, yv)], verbose=False)
        n_best = int(m.best_iteration) + 1; sv = m.predict_proba(Xv, iteration_range=(0, n_best))[:, 1]
    else:
        raise ValueError(model_name)
    thr = _pick_threshold(yv, sv)
    final = clone(base).set_params(n_estimators=n_best, learning_rate=E6_ES_LR).fit(Xtr_s, ytr)
    ys = final.predict_proba(Xte_s)[:, 1]
    return (ys >= thr).astype(int), ys, n_best

def _fp(m, Xtr, ytr, Xte):
    yp, ys = fit_predict(get_models()[m], Xtr, ytr, Xte, model_name=m)[:2]
    return yp, ys, None

def _sub_pr(y, s, mask):
    return average_precision_score(y[mask], s[mask]) if mask.sum() and len(np.unique(y[mask])) == 2 else np.nan

def paired_boot(y, s_a, s_b, B=E6_BOOT, seed=SEED):
    rs = np.random.RandomState(seed); n = len(y); d = []
    for _ in range(B):
        i = rs.randint(0, n, n)
        if y[i].sum() == 0: continue
        d.append(average_precision_score(y[i], s_b[i]) - average_precision_score(y[i], s_a[i]))
    return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

def prefix_frames(tr_c, origin_step):
    """Egitim islemleri icin as-of oznitelikler: blok [s, s+R) islemleri, adim < s yapisi ve etiketleriyle betimlenir."""
    parts, codes = [], []
    for s in range(E6_PREFIX_START, int(origin_step) + 1, E6_PREFIX_REFRESH):
        blk = tr_c[(STEP[tr_c] >= s) & (STEP[tr_c] < s + E6_PREFIX_REFRESH)]
        if len(blk) == 0: continue
        trc = tr_c[STEP[tr_c] < s]; stc = np.where(STEP < s)[0]
        if len(trc) == 0 or len(np.unique(Y[trc])) < 2: continue        # oncesinde etiketli gecmis yok
        _, fb, _ = compute_fold_features(trc, blk, stc, verbose=False)
        parts.append(fb); codes.append(blk)
    return pd.concat(parts, ignore_index=True), np.concatenate(codes)

if RUN_E6:
    _rp = os.path.join(CHECKPOINT_DIR, f'{E6_TAG}_results.csv')
    e6_rows, _done = [], set()
    if RESUME and os.path.exists(_rp):
        _prev = pd.read_csv(_rp); e6_rows = _prev.to_dict('records'); _done = set(_prev['fold'].astype(int))
        print(f'  [DEVAM] {E6_TAG}: tamamlanmis fold = {sorted(_done)}')
    E6_TRAIN_KAPPA = {}
    for fold, (tr_c, ev_c, st_c) in enumerate(PROTOCOLS['P4'][1], start=1):
        if fold in _done: continue
        t0 = time.time()
        f_tr, f_te, _ = compute_fold_features(tr_c, ev_c, st_c, verbose=False)
        p_tr, p_codes = prefix_frames(tr_c, ROLLING_ORIGINS[fold - 1])
        _pos = pd.Series(np.arange(len(tr_c)), index=tr_c)
        p_idx = _pos.loc[p_codes].values                                  # f_tr icindeki ayni satirlar
        assert (f_tr['txId'].values[p_idx] == p_tr['txId'].values).all(), 'prefix satir eslesmesi bozuk'
        y_tr, y_te, y_p = f_tr['y'].values.astype(int), f_te['y'].values.astype(int), p_tr['y'].values.astype(int)
        seen = f_te[ROUTER_SEEN_COL].values.astype(bool)
        E6_TRAIN_KAPPA[fold] = float(p_tr[ROUTER_SEEN_COL].mean())
        _o6 = {}
        for warm, cold in E6_PAIRS:
            for m in E6_MODELS:
                for key, (ftr, rows, cols, fn) in {
                        ('ref', cold): (f_tr, None, E6_SETS[cold], _fp), ('ref', warm): (f_tr, None, E6_SETS[warm], _fp),
                        ('cp', cold): (f_tr, None, E6_SETS[cold] + CP_COLS_E, _fp),
                        ('prefix', cold): (f_tr, p_idx, E6_SETS[cold], _fp), ('prefix', warm): (p_tr, None, E6_SETS[warm], _fp)}.items():
                    if (key, m) in _o6: continue
                    Xf = _X(ftr, cols) if rows is None else _X(ftr, cols)[rows]
                    yy = y_p if key[0] == 'prefix' else y_tr
                    _o6[(key, m)] = fn(m, Xf, yy, _X(f_te, cols))
                if m in E6_ES_MODELS:
                    for s in (cold, warm):
                        if (('es', s), m) not in _o6:
                            _o6[(('es', s), m)] = fit_predict_es(m, _X(f_tr, E6_SETS[s]), y_tr, _X(f_te, E6_SETS[s]))
        info = {'fold': fold, 'fold_label': FOLD_LABEL['P4'][fold], 'kappa_hat': float(seen.mean()),
                'train_kappa_prefix': E6_TRAIN_KAPPA[fold], 'n_eval': len(y_te), 'n_pos': int(y_te.sum())}
        for warm, cold in E6_PAIRS:
            for m in E6_MODELS:
                g_ = lambda v, s: _o6[((v, s), m)]
                (pa, sa, _), (pw, sw, _) = g_('ref', cold), g_('ref', warm)
                (pc, sc_, _) = g_('cp', cold); (pb, sb, _), (pp, sp, _) = g_('prefix', cold), g_('prefix', warm)
                var = {('ref', 'base'): (pa, sa, sa), ('ref', 'C'): (pw, sw, sa),
                       ('ref', 'router'): (np.where(seen, pw, pa), np.where(seen, sw, sa), sa),
                       ('cp', 'A+cp'): (pc, sc_, sa), ('cp', 'router cp'): (np.where(seen, pw, pc), np.where(seen, sw, sc_), sa),
                       ('prefix', 'base'): (pb, sb, sb), ('prefix', 'C'): (pp, sp, sb),
                       ('prefix', 'router'): (np.where(seen, pp, pb), np.where(seen, sp, sb), sb)}
                trees = {}
                if m in E6_ES_MODELS:
                    (ea, esa, na), (ew, esw, nw) = g_('es', cold), g_('es', warm)
                    var.update({('es', 'base'): (ea, esa, esa), ('es', 'C'): (ew, esw, esa),
                                ('es', 'router'): (np.where(seen, ew, ea), np.where(seen, esw, esa), esa)})
                    trees = {'n_trees_base': na, 'n_trees_warm': nw}
                for (v, role), (yp, ys, sbase) in var.items():
                    r = full_metrics(y_te, yp.astype(int), ys)
                    r.update({**info, **(trees if v == 'es' else {}), 'pair': f'{warm} | {cold}', 'model': m, 'variant': v, 'role': role,
                              'pr_auc_seen': _sub_pr(y_te, ys, seen), 'pr_auc_unseen': _sub_pr(y_te, ys, ~seen)})
                    if role != 'base':
                        r['ci_lo'], r['ci_hi'] = paired_boot(y_te, sbase, ys)
                    e6_rows.append(r)
        pd.DataFrame(e6_rows).to_csv(_rp, index=False)
        print(f'  [E6] {FOLD_LABEL["P4"][fold]} tamam ({time.time() - t0:.0f}s) | prefix egitim satiri {len(p_codes):,}/{len(tr_c):,}, '
              f'egitim kappa (prefix) = {E6_TRAIN_KAPPA[fold]:.3f}', flush=True)
        del f_tr, f_te, p_tr; gc.collect()
    E6 = pd.DataFrame(e6_rows)
    E6.to_csv(f'elliptic_e6_{PIPELINE_VERSION}.csv', index=False)
    print(f'E6: {len(E6)} satir')


Karsi taraf + islem grafi oznitelikleri (18): ['in_txs_degree', 'out_txs_degree', 'tx_pr_step', 'tx_cc_size', 'cp_seen_frac', 'cp_nsend_mean', 'cp_nsend_max', 'cp_nrecv_mean', 'cp_nrecv_max', 'cp_nsteps_mean', 'cp_nsteps_max', 'cp_btc_mean', 'cp_btc_max', 'cp_age_max', 'cp_pr_mean', 'cp_pr_max', 'cp_frate_mean', 'cp_frate_max']
  [E6] origin 24 tamam (489s) | prefix egitim satiri 21,459/23,606, egitim kappa (prefix) = 0.372
  [E6] origin 29 tamam (503s) | prefix egitim satiri 24,234/26,381, egitim kappa (prefix) = 0.375
  [E6] origin 34 tamam (552s) | prefix egitim satiri 27,747/29,894, egitim kappa (prefix) = 0.371
  [E6] origin 39 tamam (716s) | prefix egitim satiri 33,233/35,380, egitim kappa (prefix) = 0.371
  [E6] origin 44 tamam (838s) | prefix egitim satiri 40,691/42,838, egitim kappa (prefix) = 0.350
E6: 380 satir


In [38]:
# ============================================================
# E6 — TABLOLAR ve KONTROL
# ============================================================
if RUN_E6:
    pd.set_option('display.width', 250)
    if 'E5' in globals():
        for warm, cold in E6_PAIRS:
            for m in E6_MODELS:
                for role, v5 in [('base', 'base'), ('C', 'warm'), ('router', 'router raw')]:
                    a = E5[(E5.pair == f'{warm} | {cold}') & (E5.model == m) & (E5.variant == v5)].sort_values('fold')['pr_auc'].values
                    b = E6[(E6.pair == f'{warm} | {cold}') & (E6.model == m) & (E6.variant == 'ref') & (E6.role == role)].sort_values('fold')['pr_auc'].values
                    if len(a) == len(b):
                        print(f'  kontrol {m:8s} {warm[:10]:10s} {role:6s}: ref == E5  max|fark| = {np.abs(a - b).max():.2e}')
    e6_tables = {}
    E6['vr'] = E6['variant'] + ' | ' + E6['role']
    _e6_order = ['ref | base', 'ref | C', 'ref | router', 'cp | A+cp', 'cp | router cp', 'es | base', 'es | C', 'es | router',
                 'prefix | base', 'prefix | C', 'prefix | router']
    for met in ['pr_auc', 'f1', 'pr_auc_seen', 'pr_auc_unseen']:
        _p6 = E6.pivot_table(index=['pair', 'model'], columns='vr', values=met).reindex(columns=_e6_order)
        e6_tables[f'mean_{met}'] = _p6.round(4)
        print('=' * 150); print(f'E6 — {met} (origin ortalamasi)'); print('=' * 150); print(_p6.round(4).to_string()); print()
    _r6 = []
    for (pair, m, v), g6 in E6.groupby(['pair', 'model', 'variant']):
        base_role = 'base'
        for met in ['pr_auc', 'f1']:
            pv = g6.pivot_table(index='fold', columns='role', values=met)
            ref_base = E6[(E6.pair == pair) & (E6.model == m) & (E6.variant == ('ref' if v == 'cp' else v)) & (E6.role == 'base')]
            rb = ref_base.pivot_table(index='fold', values=met)[met]
            for role in [c for c in pv.columns if c != 'base']:
                dd = (pv[role] - rb).dropna()
                _r6.append({'metric': met, 'pair': pair, 'model': m, 'variant': v, 'role': role, 'ort_fark': dd.mean(),
                            'std': dd.std(ddof=1), 'kazanan_origin': f'{int((dd > 0).sum())}/{len(dd)}'})
    e6_tables['diff'] = pd.DataFrame(_r6).set_index(['metric', 'pair', 'model', 'variant', 'role']).round(4)
    print('E6 — kendi taban modeline gore fark (cp: ref taban)'); print(e6_tables['diff'].to_string()); print()
    _ci6 = E6[E6.role != 'base'][['pair', 'model', 'variant', 'role', 'fold_label', 'kappa_hat', 'ci_lo', 'ci_hi']]
    e6_tables['boot_ci'] = _ci6.round(4)
    print('E6 — eslestirilmis bootstrap %95 araligi (origin bazinda)'); print(_ci6.round(4).to_string(index=False)); print()
    _tr6 = E6[E6.variant == 'es'].groupby(['pair', 'model'])[['n_trees_base', 'n_trees_warm']].agg(['mean', 'min', 'max'])
    e6_tables['es_trees'] = _tr6.round(1)
    print('ES: secilen agac sayisi'); print(_tr6.round(1).to_string())
    print('Prefix egitim kappa:', E6.groupby('fold_label')['train_kappa_prefix'].first().round(3).to_dict())
    for _k, _t in e6_tables.items():
        tables_e[f'E6_{_k}'] = _t
    with pd.ExcelWriter('paper_tables_elliptic.xlsx') as _w:
        for _n, _t in tables_e.items():
            if _t is not None:
                _t.to_excel(_w, sheet_name=_n[:31])
    print('paper_tables_elliptic.xlsx guncellendi.')


  kontrol LR       C (B+graf) base  : ref == E5  max|fark| = 5.55e-17
  kontrol LR       C (B+graf) C     : ref == E5  max|fark| = 5.55e-17
  kontrol LR       C (B+graf) router: ref == E5  max|fark| = 9.71e-17
  kontrol RF       C (B+graf) base  : ref == E5  max|fark| = 1.11e-16
  kontrol RF       C (B+graf) C     : ref == E5  max|fark| = 1.11e-16
  kontrol RF       C (B+graf) router: ref == E5  max|fark| = 1.11e-16
  kontrol XGBoost  C (B+graf) base  : ref == E5  max|fark| = 0.00e+00
  kontrol XGBoost  C (B+graf) C     : ref == E5  max|fark| = 1.11e-16
  kontrol XGBoost  C (B+graf) router: ref == E5  max|fark| = 1.11e-16
  kontrol LightGBM C (B+graf) base  : ref == E5  max|fark| = 1.11e-16
  kontrol LightGBM C (B+graf) C     : ref == E5  max|fark| = 1.11e-16
  kontrol LightGBM C (B+graf) router: ref == E5  max|fark| = 1.11e-16
  kontrol LR       A+AGG+C    base  : ref == E5  max|fark| = 9.71e-17
  kontrol LR       A+AGG+C    C     : ref == E5  max|fark| = 5.55e-17
  kontrol LR       A